[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/linear_algebra/08_numerical_linear_algebra_iterative_solvers/exercises.ipynb)

# Module 08 — Exercises: Numerical Linear Algebra and Iterative Solvers

Forty-three solved problems in four tiers. Every problem carries a statement, a short intuition,
a stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is numeric or
algorithmic — a code cell that recomputes it.

Theorem, lemma and proof numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): transposes written $A^{\top}$, norms written
$\lVert \cdot \rVert$, eigenvalues descending, $T_k$ the Chebyshev polynomial of the first kind,
$u$ the unit roundoff and $\gamma_n = nu/(1-nu)$.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps
U = EPS / 2
print(f"eps_mach = {EPS:.6e}    unit roundoff u = {U:.6e}")

eps_mach = 2.220446e-16    unit roundoff u = 1.110223e-16


## L0 — Concept Checks

### Problem L0.1 — Unit roundoff of a binary format

**Statement.** A binary floating-point format has base $\beta = 2$ and $t = 24$ significand
bits, the hidden bit included, with round to nearest. Give its unit roundoff $u$.

**Intuition.** The representable numbers just above $1$ are spaced $\beta^{1-t}$ apart, and
rounding to nearest never moves a number by more than half a gap.

**Solution.**

*Step 1.* The gap between $1$ and its successor is $\varepsilon_{\mathrm{mach}} = \beta^{1-t}$.

*Step 2.* Round to nearest costs at most half of that, so $u = \tfrac12 \beta^{1-t} = \beta^{-t}$
for $\beta = 2$.

*Step 3.* With $t = 24$: $u = 2^{-24} \approx 5.96 \times 10^{-8}$.

$$
\boxed{u = 2^{-24} \approx 5.96 \times 10^{-8}}
$$

**Key takeaway.** The significand length alone fixes the relative accuracy of every arithmetic
operation, through Definition 3.1.

In [2]:
t = 24
u_single = 2.0 ** (-t)
print(f"u for t = {t} bits : {u_single:.6e}")
print(f"numpy float32 eps  : {np.finfo(np.float32).eps:.6e}  = 2^{np.log2(np.finfo(np.float32).eps):.0f}")
print(f"half that          : {np.finfo(np.float32).eps/2:.6e}")
assert abs(u_single - np.finfo(np.float32).eps / 2) < 1e-20

u for t = 24 bits : 5.960464e-08
numpy float32 eps  : 1.192093e-07  = 2^-23
half that          : 5.960464e-08


### Problem L0.2 — Flop count of a sparse matrix-vector product

**Statement.** How many floating-point operations does $y = Ax$ take when
$A \in \mathbb{R}^{n \times n}$ has $k = \operatorname{nnz}(A)$ non-zero entries?

**Intuition.** Zero entries contribute nothing, so only the stored entries do arithmetic.

**Solution.**

*Step 1.* $y_i = \sum_{j : a_{ij} \neq 0} a_{ij}x_j$.

*Step 2.* Each stored entry costs one multiplication and one addition into the running sum.

*Step 3.* Summing over all $k$ stored entries gives $2k$ flops.

$$
\boxed{2k \text{ flops, independent of } n}
$$

**Key takeaway.** This is why iterative solvers scale: their unit of work is
$\operatorname{nnz}(A)$, while a dense factorization is $\tfrac23 n^3$.

In [3]:
n_s = 2000
band = np.zeros((n_s, n_s))
idx = np.arange(n_s)
band[idx, idx] = 2.0
band[idx[:-1], idx[1:]] = -1.0
band[idx[1:], idx[:-1]] = -1.0
nnz = int(np.count_nonzero(band))
print(f"n = {n_s},  nnz = {nnz},  SpMV flops = {2*nnz},  dense MatVec flops = {2*n_s**2 - n_s}")
print(f"dense LU flops ~ {2*n_s**3/3:.3e},  ratio to SpMV = {(2*n_s**3/3)/(2*nnz):.3e}")
assert nnz == 3 * n_s - 2

n = 2000,  nnz = 5998,  SpMV flops = 11996,  dense MatVec flops = 7998000
dense LU flops ~ 5.333e+09,  ratio to SpMV = 4.446e+05


### Problem L0.3 — Condition number of an orthogonal matrix

**Statement.** Compute $\kappa_2(Q)$ for $Q^{\top}Q = I$.

**Intuition.** An orthogonal map is a rotation or reflection: it changes no length, so it can
neither amplify nor damp a relative error.

**Solution.**

*Step 1.* $\lVert Qx \rVert_2^2 = x^{\top}Q^{\top}Qx = \lVert x \rVert_2^2$, so
$\lVert Q \rVert_{\mathrm{op}} = 1$.

*Step 2.* $Q^{-1} = Q^{\top}$ is also orthogonal, so $\lVert Q^{-1} \rVert_{\mathrm{op}} = 1$.

*Step 3.* $\kappa_2(Q) = 1 \cdot 1 = 1$, the minimum possible value by Definition 3.3.

$$
\boxed{\kappa_2(Q) = 1}
$$

**Key takeaway.** Orthogonal transformations are the only perfectly conditioned ones, which is
why QR and Householder reflections are the backbone of stable dense linear algebra.

In [4]:
Qq, _ = np.linalg.qr(rng.standard_normal((6, 6)))
print("||Q^T Q - I||_F :", np.linalg.norm(Qq.T @ Qq - np.eye(6)))
print("||Q||_2         :", np.linalg.norm(Qq, 2))
print("||Q^-1||_2      :", np.linalg.norm(np.linalg.inv(Qq), 2))
print("kappa_2(Q)      :", np.linalg.cond(Qq))
assert abs(np.linalg.cond(Qq) - 1.0) < 1e-12

||Q^T Q - I||_F : 7.828542519733557e-16
||Q||_2         : 1.0000000000000002
||Q^-1||_2      : 1.0000000000000002
kappa_2(Q)      : 1.0000000000000002


### Problem L0.4 — Largest possible Krylov dimension

**Statement.** What is the largest possible value of $\dim \mathcal{K}_k(A, v)$ for
$A \in \mathbb{R}^{n \times n}$ and $v \neq 0$?

**Intuition.** The vectors live in $\mathbb{R}^n$, and Cayley-Hamilton makes $A^n v$ redundant.

**Solution.**

*Step 1.* $\mathcal{K}_k(A,v) \subseteq \mathbb{R}^n$, so its dimension is at most $n$.

*Step 2.* Cayley-Hamilton gives $p_A(A) = 0$ with $\deg p_A = n$, so $A^{n}v$ is a combination of
$v, Av, \dots, A^{n-1}v$ and the sequence stops growing at $n$ terms.

*Step 3.* The bound $n$ is attained, for instance by a companion matrix with $v = e_1$.

$$
\boxed{\dim \mathcal{K}_k(A,v) \le \min(k, n) \le n}
$$

**Key takeaway.** Every Krylov method terminates in at most $n$ steps in exact arithmetic —
Theorem 4.6 statement 5 for CG, and the same count for GMRES.

In [5]:
n_k = 6
C = np.zeros((n_k, n_k))
C[1:, :-1] = np.eye(n_k - 1)
C[:, -1] = rng.standard_normal(n_k)
v = np.zeros(n_k)
v[0] = 1.0
K = np.column_stack([np.linalg.matrix_power(C, j) @ v for j in range(n_k + 2)])
for k in (1, 3, 6, 8):
    print(f"dim K_{k} = {np.linalg.matrix_rank(K[:, :k])}")
assert np.linalg.matrix_rank(K) == n_k

dim K_1 = 1
dim K_3 = 3
dim K_6 = 6
dim K_8 = 6


### Problem L0.5 — Norms and condition number of a diagonal matrix

**Statement.** For $A = \operatorname{diag}(3, -4)$ compute
$\lVert A \rVert_{\mathrm{op}}$, $\lVert A^{-1} \rVert_{\mathrm{op}}$ and $\kappa_2(A)$.

**Intuition.** A diagonal matrix stretches each axis independently, so its singular values are
the absolute diagonal entries.

**Solution.**

*Step 1.* Singular values: $\sigma_1 = 4$, $\sigma_2 = 3$, so $\lVert A \rVert_{\mathrm{op}} = 4$.

*Step 2.* $A^{-1} = \operatorname{diag}(1/3, -1/4)$ has largest singular value $1/3$.

*Step 3.* $\kappa_2(A) = 4 \cdot \tfrac13 = \tfrac43$.

$$
\boxed{\lVert A \rVert_{\mathrm{op}} = 4, \quad \lVert A^{-1} \rVert_{\mathrm{op}} = \tfrac13, \quad \kappa_2(A) = \tfrac43}
$$

**Key takeaway.** For any diagonal matrix $\kappa_2 = \max_i \lvert a_{ii} \rvert / \min_i \lvert a_{ii} \rvert$;
the sign is irrelevant.

In [6]:
Ad = np.diag([3.0, -4.0])
print("||A||_2    :", np.linalg.norm(Ad, 2))
print("||A^-1||_2 :", np.linalg.norm(np.linalg.inv(Ad), 2))
print("kappa_2(A) :", np.linalg.cond(Ad), " 4/3 =", 4 / 3)
assert abs(np.linalg.norm(Ad, 2) - 4) < 1e-12
assert abs(np.linalg.cond(Ad) - 4 / 3) < 1e-12

||A||_2    : 4.0
||A^-1||_2 : 0.3333333333333333
kappa_2(A) : 1.3333333333333333  4/3 = 1.3333333333333333


### Problem L0.6 — The Gauss-Seidel splitting

**Statement.** With $A = D - L - U$ as in Definition 3.4, write the Gauss-Seidel iteration in
the form $x_{k+1} = Gx_k + c$ and identify $G$.

**Intuition.** Gauss-Seidel uses each freshly computed component immediately, which amounts to
inverting the whole lower triangle at once.

**Solution.**

*Step 1.* Update rule: $(D - L)x_{k+1} = Ux_k + b$.

*Step 2.* $D - L$ is lower triangular with the diagonal of $A$ on its diagonal, hence invertible
when every $a_{ii} \neq 0$.

*Step 3.* Solving, $x_{k+1} = (D-L)^{-1}Ux_k + (D-L)^{-1}b$.

$$
\boxed{G_{\mathrm{GS}} = (D-L)^{-1}U, \qquad c = (D-L)^{-1}b}
$$

**Key takeaway.** The splitting is $M = D - L$, $N = U$; the cost per step is one triangular
solve rather than one diagonal solve.

In [7]:
A_g = np.array([[4.0, -1.0, 0.0], [-1.0, 4.0, -1.0], [0.0, -1.0, 4.0]])
D_g = np.diag(np.diag(A_g))
L_g = -np.tril(A_g, -1)
U_g = -np.triu(A_g, 1)
print("D - L - U == A :", np.allclose(D_g - L_g - U_g, A_g))
G_g = np.linalg.solve(D_g - L_g, U_g)
b_g = np.array([1.0, 2.0, 3.0])
c_g = np.linalg.solve(D_g - L_g, b_g)
x = np.zeros(3)
for _ in range(60):
    x = G_g @ x + c_g
print("fixed point   :", x)
print("direct solve  :", np.linalg.solve(A_g, b_g))
assert np.allclose(x, np.linalg.solve(A_g, b_g))

D - L - U == A : True
fixed point   : [0.4643 0.8571 0.9643]
direct solve  : [0.4643 0.8571 0.9643]


### Problem L0.7 — Residual against error

**Statement.** With $Ax = b$ and an approximation $x_k$, relate $e_k = x - x_k$ to
$r_k = b - Ax_k$ and bound $\lVert e_k \rVert$ by $\lVert r_k \rVert$.

**Intuition.** The residual is what you can measure; the error is what you want. They differ by
one application of $A^{-1}$.

**Solution.**

*Step 1.* $r_k = b - Ax_k = Ax - Ax_k = Ae_k$.

*Step 2.* Hence $e_k = A^{-1}r_k$.

*Step 3.* Taking norms, $\lVert e_k \rVert \le \lVert A^{-1} \rVert \lVert r_k \rVert$.

$$
\boxed{r_k = Ae_k, \qquad \lVert e_k \rVert \le \lVert A^{-1} \rVert \, \lVert r_k \rVert}
$$

**Key takeaway.** A small residual certifies a small error only when $\lVert A^{-1} \rVert$ is
small; this is Theorem 4.1 in disguise, and the reason stopping criteria are written relative to
$\lVert b \rVert$.

In [8]:
A_r = np.array([[1.0, 1.0], [1.0, 1.0 + 1e-6]])
x_true = np.array([1.0, 1.0])
b_r = A_r @ x_true
x_k = x_true + np.array([1e-3, -1e-3])
r_k = b_r - A_r @ x_k
e_k = x_true - x_k
print("kappa_2(A)       :", np.linalg.cond(A_r))
print("||r_k||          :", np.linalg.norm(r_k))
print("||e_k||          :", np.linalg.norm(e_k))
print("||A^-1|| ||r_k|| :", np.linalg.norm(np.linalg.inv(A_r), 2) * np.linalg.norm(r_k))
assert np.allclose(r_k, A_r @ e_k)
assert np.linalg.norm(e_k) <= np.linalg.norm(np.linalg.inv(A_r), 2) * np.linalg.norm(r_k) + 1e-14

kappa_2(A)       : 4000002.0003309543
||r_k||          : 1.000000082740371e-09
||e_k||          : 0.0014142135623730178
||A^-1|| ||r_k|| : 0.0020000006656454416


### Problem L0.8 — The Jacobi iteration matrix of a $2 \times 2$

**Statement.** Compute $G_{\mathrm{J}}$ for $A = \left(\begin{smallmatrix}2&1\\1&2\end{smallmatrix}\right)$.

**Intuition.** Jacobi divides each row by its diagonal entry and negates what is left.

**Solution.**

*Step 1.* $D = 2I$ and $L + U = -\left(\begin{smallmatrix}0&1\\1&0\end{smallmatrix}\right)$
in the convention $A = D - L - U$.

*Step 2.* $G_{\mathrm{J}} = D^{-1}(L+U) = \tfrac12 \cdot \left(-\left(\begin{smallmatrix}0&1\\1&0\end{smallmatrix}\right)\right)$.

$$
\boxed{G_{\mathrm{J}} = \begin{pmatrix} 0 & -0.5 \\ -0.5 & 0 \end{pmatrix}}
$$

**Key takeaway.** Off-diagonal entry $a_{ij}$ becomes $-a_{ij}/a_{ii}$, which is why row
diagonal dominance is exactly the statement $\lVert G_{\mathrm{J}} \rVert_\infty \lt 1$
(Proof 5.3).

In [9]:
A_j = np.array([[2.0, 1.0], [1.0, 2.0]])
D_j = np.diag(np.diag(A_j))
L_j = -np.tril(A_j, -1)
U_j = -np.triu(A_j, 1)
G_j = np.linalg.solve(D_j, L_j + U_j)
print("G_Jacobi:\n", G_j)
print("||G_J||_inf :", np.abs(G_j).sum(axis=1).max())
assert np.allclose(G_j, [[0, -0.5], [-0.5, 0]])

G_Jacobi:
 [[ 0.  -0.5]
 [-0.5  0. ]]
||G_J||_inf : 0.5


### Problem L0.9 — Spectral radius and convergence

**Statement.** Find $\rho(G_{\mathrm{J}})$ for
$G_{\mathrm{J}} = \left(\begin{smallmatrix}0&-0.5\\-0.5&0\end{smallmatrix}\right)$ and say
whether Jacobi converges.

**Intuition.** Convergence of a stationary method is decided entirely by the largest eigenvalue
modulus of its iteration matrix.

**Solution.**

*Step 1.* $\det(\lambda I - G_{\mathrm{J}}) = \lambda^2 - \tfrac14$, so $\lambda = \pm \tfrac12$.

*Step 2.* $\rho(G_{\mathrm{J}}) = \tfrac12 \lt 1$.

*Step 3.* By Theorem 4.2 the iteration converges from every $x_0$, with asymptotic contraction
factor $\tfrac12$ per step.

$$
\boxed{\rho(G_{\mathrm{J}}) = 0.5 \lt 1, \text{ so Jacobi converges}}
$$

**Key takeaway.** $\rho \lt 1$ is necessary and sufficient; the number itself is the per-step
error reduction, so $-\ln \rho$ digits are gained per iteration.

In [10]:
G_j = np.array([[0.0, -0.5], [-0.5, 0.0]])
print("eigenvalues :", np.linalg.eigvals(G_j))
print("rho(G_J)    :", max(abs(np.linalg.eigvals(G_j))))
e = np.array([1.0, 0.3])
for k in (1, 5, 10, 20):
    print(f"  ||G^{k} e|| / ||e|| = {np.linalg.norm(np.linalg.matrix_power(G_j, k) @ e)/np.linalg.norm(e):.6e}"
          f"   0.5^{k} = {0.5**k:.6e}")
assert abs(max(abs(np.linalg.eigvals(G_j))) - 0.5) < 1e-12

eigenvalues : [ 0.5 -0.5]
rho(G_J)    : 0.5000000000000001
  ||G^1 e|| / ||e|| = 5.000000e-01   0.5^1 = 5.000000e-01
  ||G^5 e|| / ||e|| = 3.125000e-02   0.5^5 = 3.125000e-02
  ||G^10 e|| / ||e|| = 9.765625e-04   0.5^10 = 9.765625e-04
  ||G^20 e|| / ||e|| = 9.536743e-07   0.5^20 = 9.536743e-07


## L1 — Foundations

### Problem L1.1 — Strict diagonal dominance makes Jacobi converge

**Statement.** Prove that if $A$ is strictly row diagonally dominant, that is
$\lvert a_{ii} \rvert \gt \sum_{j \neq i} \lvert a_{ij} \rvert$ for every $i$, then Jacobi
converges from every $x_0$.

**Intuition.** Dominance says each row's diagonal outweighs the rest of the row, so dividing by
it shrinks everything else.

**Solution.**

*Step 1.* Dominance forces $a_{ii} \neq 0$, so $D$ is invertible and
$G_{\mathrm{J}} = D^{-1}(L+U)$ exists.

*Step 2.* Its entries are $(G_{\mathrm{J}})_{ij} = -a_{ij}/a_{ii}$ for $i \neq j$, and $0$ on the
diagonal.

*Step 3.* The induced $\infty$-norm is the maximum absolute row sum:

$$
\lVert G_{\mathrm{J}} \rVert_\infty = \max_i \sum_{j \neq i} \frac{\lvert a_{ij} \rvert}{\lvert a_{ii} \rvert} \lt 1 ,
$$

by dominance applied row by row.

*Step 4.* $\rho(G_{\mathrm{J}}) \le \lVert G_{\mathrm{J}} \rVert_\infty \lt 1$, because every
eigenvalue is bounded by any induced norm. Theorem 4.2 finishes the argument.

$$
\boxed{\lVert G_{\mathrm{J}} \rVert_\infty \lt 1 \implies \rho(G_{\mathrm{J}}) \lt 1 \implies \text{Jacobi converges}}
$$

**Key takeaway.** Dominance is sufficient but not necessary, and it cannot be traded for positive
definiteness: Section 7.4 of the theory notebook runs a positive definite matrix with
$\rho(G_{\mathrm{J}}) = 1.8$.

In [11]:
A_dd = np.array([[10.0, -1.0, 2.0], [-1.0, 11.0, -1.0], [2.0, -1.0, 10.0]])
margins = [abs(A_dd[i, i]) - (np.abs(A_dd[i]).sum() - abs(A_dd[i, i])) for i in range(3)]
D_dd = np.diag(np.diag(A_dd))
G_dd = np.linalg.solve(D_dd, -(A_dd - D_dd))
print("dominance margins   :", np.round(margins, 4))
print("||G_J||_inf         :", np.abs(G_dd).sum(axis=1).max())
print("rho(G_J)            :", max(abs(np.linalg.eigvals(G_dd))))
x_dd = np.zeros(3)
b_dd = A_dd @ np.array([1.0, 2.0, -1.0])
for k in range(1, 41):
    x_dd = G_dd @ x_dd + np.linalg.solve(D_dd, b_dd)
    if k in (5, 20, 40):
        print(f"  k = {k:2d}  error = {np.linalg.norm(x_dd - [1, 2, -1]):.3e}")
assert all(m > 0 for m in margins)
assert np.abs(G_dd).sum(axis=1).max() < 1.0

dominance margins   : [7. 9. 7.]
||G_J||_inf         : 0.30000000000000004
rho(G_J)            : 0.2678744119329038
  k =  5  error = 1.366e-03
  k = 20  error = 3.382e-12
  k = 40  error = 0.000e+00


### Problem L1.2 — The Thomas algorithm on a $4 \times 4$ tridiagonal system

**Statement.** Solve $Ax = b$ with $A = \operatorname{tridiag}(-1, 4, -1) \in \mathbb{R}^{4\times4}$
and $b = (5,5,5,5)^{\top}$ by the Thomas algorithm. Count the flops for general $n$, and prove
that strict diagonal dominance keeps every multiplier below $1$ in modulus, so no pivoting is
needed.

**Intuition.** Gaussian elimination on a tridiagonal matrix touches only one subdiagonal entry
per row, so the whole factorization collapses to two scalar recurrences.

**Solution.**

*Step 1 — the recurrences.* With $a_i$ the subdiagonal, $b_i$ the diagonal and $c_i$ the
superdiagonal,

$$
c'_1 = \frac{c_1}{b_1}, \quad d'_1 = \frac{d_1}{b_1}, \qquad
c'_i = \frac{c_i}{b_i - a_i c'_{i-1}}, \quad d'_i = \frac{d_i - a_i d'_{i-1}}{b_i - a_i c'_{i-1}} ,
$$

followed by $x_n = d'_n$ and $x_i = d'_i - c'_i x_{i+1}$.

*Step 2 — run it.* With $a_i = c_i = -1$, $b_i = 4$, $d_i = 5$:

| $i$ | denominator $b_i - a_i c'_{i-1}$ | $c'_i$ | $d'_i$ |
|---:|---:|---:|---:|
| 1 | $4$ | $-0.250000$ | $1.250000$ |
| 2 | $3.750000$ | $-0.266667$ | $1.666667$ |
| 3 | $3.733333$ | $-0.267857$ | $1.785714$ |
| 4 | $3.732143$ | — | $1.818182$ |

*Step 3 — back-substitute.* $x_4 = 1.818182$, then
$x_3 = 1.785714 + 0.267857 x_4 = 2.272727$, $x_2 = 2.272727$, $x_1 = 1.818182$. In fractions,
$x = \tfrac{1}{11}(20, 25, 25, 20)^{\top}$.

*Step 4 — flop count.* The first row costs $2$; each of rows $2$ to $n-1$ costs $6$ and row $n$
costs $5$; back-substitution costs $2(n-1)$. Total $8n - 7$, which is $25$ for $n = 4$.

*Step 5 — bounded multipliers.* Claim $\lvert c'_i \rvert \lt 1$ for all $i$, by induction.
For $i = 1$, $\lvert c'_1 \rvert = \lvert c_1 \rvert / \lvert b_1 \rvert \lt 1$ by dominance.
If $\lvert c'_{i-1} \rvert \lt 1$ then

$$
\lvert b_i - a_i c'_{i-1} \rvert \ \ge\ \lvert b_i \rvert - \lvert a_i \rvert \, \lvert c'_{i-1} \rvert \ \gt\ \lvert b_i \rvert - \lvert a_i \rvert \ \gt\ \lvert c_i \rvert ,
$$

so $\lvert c'_i \rvert \lt 1$ and, in particular, the denominator never vanishes.

$$
\boxed{x = \tfrac{1}{11}(20, 25, 25, 20)^{\top}, \qquad 8n - 7 \text{ flops}, \qquad \lvert c'_i \rvert \lt 1}
$$

**Key takeaway.** For a diagonally dominant tridiagonal system the direct solve is $O(n)$ and
needs no pivoting, so it is the one case where a direct method beats every iterative method.

In [12]:
def thomas(a, bdiag, c, d):
    """Thomas algorithm for a tridiagonal system; a and c have length n-1."""
    n_ = len(bdiag)
    cp = np.zeros(n_ - 1)
    dp = np.zeros(n_)
    cp[0] = c[0] / bdiag[0]
    dp[0] = d[0] / bdiag[0]
    dens = [bdiag[0]]
    for i in range(1, n_):
        den = bdiag[i] - a[i - 1] * cp[i - 1] if i > 0 else bdiag[i]
        dens.append(den)
        if i < n_ - 1:
            cp[i] = c[i] / den
        dp[i] = (d[i] - a[i - 1] * dp[i - 1]) / den
    x = np.zeros(n_)
    x[-1] = dp[-1]
    for i in range(n_ - 2, -1, -1):
        x[i] = dp[i] - cp[i] * x[i + 1]
    return x, np.array(cp), np.array(dp), np.array(dens)


n_t = 4
a_t = -np.ones(n_t - 1)
b_t = 4 * np.ones(n_t)
c_t = -np.ones(n_t - 1)
d_t = 5 * np.ones(n_t)
x_t, cp, dp, dens = thomas(a_t, b_t, c_t, d_t)
A_t = np.diag(b_t) + np.diag(a_t, -1) + np.diag(c_t, 1)
print("denominators :", np.round(dens, 6))
print("c'           :", np.round(cp, 6))
print("d'           :", np.round(dp, 6))
print("x (Thomas)   :", np.round(x_t, 6))
print("x (exact)    :", np.round(np.linalg.solve(A_t, d_t), 6), " = (20,25,25,20)/11")
print("residual     :", np.linalg.norm(A_t @ x_t - d_t))
print(f"flops 8n - 7 = {8*n_t - 7}   dense LU would be about {2*n_t**3/3:.0f}")
assert np.allclose(x_t, np.array([20, 25, 25, 20]) / 11)
assert np.all(np.abs(cp) < 1.0)

denominators : [4.     3.75   3.7333 3.7321]
c'           : [-0.25   -0.2667 -0.2679]
d'           : [1.25   1.6667 1.7857 1.8182]
x (Thomas)   : [1.8182 2.2727 2.2727 1.8182]
x (exact)    : [1.8182 2.2727 2.2727 1.8182]  = (20,25,25,20)/11
residual     : 8.881784197001252e-16
flops 8n - 7 = 25   dense LU would be about 43


### Problem L1.3 — The Gauss-Seidel iteration matrix of a $2 \times 2$

**Statement.** Compute $G_{\mathrm{GS}}$ for
$A = \left(\begin{smallmatrix}2&1\\1&2\end{smallmatrix}\right)$.

**Intuition.** One triangular solve replaces the diagonal solve of Jacobi.

**Solution.**

*Step 1.* $D - L = \left(\begin{smallmatrix}2&0\\1&2\end{smallmatrix}\right)$ and
$U = \left(\begin{smallmatrix}0&-1\\0&0\end{smallmatrix}\right)$.

*Step 2.* $(D-L)^{-1} = \tfrac14\left(\begin{smallmatrix}2&0\\-1&2\end{smallmatrix}\right) = \left(\begin{smallmatrix}0.5&0\\-0.25&0.5\end{smallmatrix}\right)$.

*Step 3.* Multiplying, $G_{\mathrm{GS}} = (D-L)^{-1}U = \left(\begin{smallmatrix}0&-0.5\\0&0.25\end{smallmatrix}\right)$.

$$
\boxed{G_{\mathrm{GS}} = \begin{pmatrix} 0 & -0.5 \\ 0 & 0.25 \end{pmatrix}}
$$

**Key takeaway.** The first column vanishes because the first component is already exact after
its own update — Gauss-Seidel propagates information within the sweep.

In [13]:
A_gs = np.array([[2.0, 1.0], [1.0, 2.0]])
D_gs = np.diag(np.diag(A_gs))
L_gs = -np.tril(A_gs, -1)
U_gs = -np.triu(A_gs, 1)
G_gs = np.linalg.solve(D_gs - L_gs, U_gs)
print("(D - L)^-1:\n", np.linalg.inv(D_gs - L_gs))
print("G_GS      :\n", G_gs)
assert np.allclose(G_gs, [[0.0, -0.5], [0.0, 0.25]])

(D - L)^-1:
 [[ 0.5   0.  ]
 [-0.25  0.5 ]]
G_GS      :
 [[-0.   -0.5 ]
 [ 0.    0.25]]


### Problem L1.4 — Gauss-Seidel is exactly twice as fast here

**Statement.** Compute $\rho(G_{\mathrm{GS}})$ for
$G_{\mathrm{GS}} = \left(\begin{smallmatrix}0&-0.5\\0&0.25\end{smallmatrix}\right)$ and compare
it with $\rho(G_{\mathrm{J}}) = 0.5$ from Problem L0.9.

**Intuition.** For a consistently ordered matrix, one Gauss-Seidel sweep does the work of two
Jacobi sweeps.

**Solution.**

*Step 1.* $G_{\mathrm{GS}}$ is upper triangular, so its eigenvalues are $0$ and $0.25$.

*Step 2.* $\rho(G_{\mathrm{GS}}) = 0.25$.

*Step 3.* $0.25 = 0.5^2 = \rho(G_{\mathrm{J}})^2$, so the asymptotic rates satisfy
$R_\infty(G_{\mathrm{GS}}) = 2R_\infty(G_{\mathrm{J}})$ and Gauss-Seidel needs half as many
iterations.

$$
\boxed{\rho(G_{\mathrm{GS}}) = 0.25 = \rho(G_{\mathrm{J}})^{2}}
$$

**Key takeaway.** The squaring relation holds for consistently ordered matrices (Young's theory,
Problem L3.6), not for every matrix; the theory notebook's Section 7.4 shows a case where
Jacobi diverges and Gauss-Seidel does not.

In [14]:
G_gs = np.array([[0.0, -0.5], [0.0, 0.25]])
G_j = np.array([[0.0, -0.5], [-0.5, 0.0]])
r_gs = max(abs(np.linalg.eigvals(G_gs)))
r_j = max(abs(np.linalg.eigvals(G_j)))
print(f"rho(G_GS) = {r_gs:.6f}   rho(G_J)^2 = {r_j**2:.6f}")
print("iterations for 1e-8:", np.ceil(np.log(1e-8) / np.log(r_gs)), "vs",
      np.ceil(np.log(1e-8) / np.log(r_j)))
assert abs(r_gs - r_j ** 2) < 1e-12

rho(G_GS) = 0.250000   rho(G_J)^2 = 0.250000
iterations for 1e-8: 14.0 vs 27.0


### Problem L1.5 — Kahan's necessary condition for SOR

**Statement.** Prove that SOR can converge only if $0 \lt \omega \lt 2$.

**Intuition.** The determinant of the SOR iteration matrix is $(1-\omega)^n$ whatever $A$ is, and
a matrix whose eigenvalue product has modulus $\ge 1$ cannot have all eigenvalues inside the unit
disc.

**Solution.**

*Step 1.* $G_\omega = (\omega^{-1}D - L)^{-1}\bigl((\omega^{-1}-1)D + U\bigr)$, and both factors
are triangular.

*Step 2.* Hence $\det(\omega^{-1}D - L) = \omega^{-n}\det D$ and
$\det((\omega^{-1}-1)D + U) = (\omega^{-1}-1)^n \det D$.

*Step 3.* Dividing, $\det G_\omega = (1-\omega)^n$.

*Step 4.* The determinant is the product of the eigenvalues, so
$\rho(G_\omega)^n \ge \prod_i \lvert \lambda_i \rvert = \lvert 1-\omega \rvert^n$, giving
$\rho(G_\omega) \ge \lvert 1 - \omega \rvert$.

*Step 5.* Convergence needs $\rho(G_\omega) \lt 1$, hence $\lvert 1-\omega \rvert \lt 1$.

$$
\boxed{\rho(G_\omega) \ge \lvert 1-\omega \rvert, \text{ so convergence requires } 0 \lt \omega \lt 2}
$$

**Key takeaway.** This is Theorem 4.4. It is only necessary; sufficiency for symmetric positive
definite $A$ is Ostrowski-Reich, Theorem 4.5.

In [15]:
A_w = np.array([[4.0, -1.0, 0.0], [-1.0, 4.0, -1.0], [0.0, -1.0, 4.0]])
D_w = np.diag(np.diag(A_w))
L_w = -np.tril(A_w, -1)
U_w = -np.triu(A_w, 1)
print("  omega    det(G_omega)    (1-omega)^n     rho(G_omega)   |1-omega|")
for w in (0.4, 1.0, 1.3, 1.7, 2.0, 2.4):
    Gw = np.linalg.solve(D_w / w - L_w, (1 / w - 1) * D_w + U_w)
    rw = max(abs(np.linalg.eigvals(Gw)))
    print(f"  {w:5.2f}   {np.linalg.det(Gw):12.6f}   {(1-w)**3:12.6f}   {rw:10.6f}   {abs(1-w):.6f}")
    assert rw >= abs(1 - w) - 1e-12
    assert abs(np.linalg.det(Gw) - (1 - w) ** 3) < 1e-10

  omega    det(G_omega)    (1-omega)^n     rho(G_omega)   |1-omega|
   0.40       0.216000       0.216000     0.720000   0.600000
   1.00       0.000000       0.000000     0.125000   0.000000
   1.30      -0.027000      -0.027000     0.300000   0.300000
   1.70      -0.343000      -0.343000     0.700000   0.700000
   2.00      -1.000000      -1.000000     1.000000   1.000000
   2.40      -2.744000      -2.744000     1.400000   1.400000


### Problem L1.6 — The optimal relaxation parameter

**Statement.** For a consistently ordered matrix with $\mu = \rho(G_{\mathrm{J}}) = 0.5$, find
$\omega_{\mathrm{opt}}$ and $\rho(G_{\omega_{\mathrm{opt}}})$.

**Intuition.** Young's relation makes the two roots of a quadratic collide exactly at the optimal
$\omega$, and a double root is where the spectral radius bottoms out.

**Solution.**

*Step 1.* Young's formula: $\omega_{\mathrm{opt}} = 2/(1 + \sqrt{1-\mu^2})$.

*Step 2.* $\mu^2 = 0.25$, so $\sqrt{1-\mu^2} = \sqrt{3}/2$.

*Step 3.* $\omega_{\mathrm{opt}} = \dfrac{2}{1 + \sqrt3/2} = \dfrac{4}{2+\sqrt3} = 4(2-\sqrt3) = 8 - 4\sqrt3 \approx 1.07180$.

*Step 4.* At the optimum $\rho(G_{\omega_{\mathrm{opt}}}) = \omega_{\mathrm{opt}} - 1 = 7 - 4\sqrt3 \approx 0.07180$.

$$
\boxed{\omega_{\mathrm{opt}} = 8 - 4\sqrt3 \approx 1.07180, \qquad \rho = 7 - 4\sqrt3 \approx 0.07180}
$$

**Key takeaway.** Relaxation turns $\rho = 0.5$ into $\rho = 0.0718$ for the price of one
scalar, so each iteration gains $3.8$ times as many digits. The derivation of Young's relation is
Problem L3.6.

In [16]:
mu = 0.5
w_opt = 2 / (1 + np.sqrt(1 - mu ** 2))
print(f"omega_opt         = {w_opt:.6f}   8 - 4 sqrt3 = {8 - 4*np.sqrt(3):.6f}")
print(f"rho at optimum    = {w_opt - 1:.6f}   7 - 4 sqrt3 = {7 - 4*np.sqrt(3):.6f}")
A_o = np.array([[2.0, 1.0], [1.0, 2.0]])
D_o = np.diag(np.diag(A_o))
L_o = -np.tril(A_o, -1)
U_o = -np.triu(A_o, 1)
best = min(((max(abs(np.linalg.eigvals(np.linalg.solve(D_o / w - L_o, (1 / w - 1) * D_o + U_o)))), w)
            for w in np.linspace(0.5, 1.9, 15001)))
print(f"numerical minimum : rho = {best[0]:.6f} at omega = {best[1]:.6f}")
print(f"digits per iteration: Jacobi {-np.log10(mu):.4f}, SOR {-np.log10(w_opt-1):.4f}, "
      f"ratio {np.log10(w_opt-1)/np.log10(mu):.4f}")
assert abs(w_opt - (8 - 4 * np.sqrt(3))) < 1e-12
assert abs(best[1] - w_opt) < 1e-3 and abs(best[0] - (w_opt - 1)) < 1e-4

omega_opt         = 1.071797   8 - 4 sqrt3 = 1.071797
rho at optimum    = 0.071797   7 - 4 sqrt3 = 0.071797


numerical minimum : rho = 0.071853 at omega = 1.071853
digits per iteration: Jacobi 0.3010, SOR 1.1439, ratio 3.7999


### Problem L1.7 — Condition number of the 1-D discrete Laplacian

**Statement.** Find the asymptotic behaviour of $\kappa_2(A)$ for
$A = \operatorname{tridiag}(-1,2,-1) \in \mathbb{R}^{N \times N}$ as $N \to \infty$.

**Intuition.** The smallest eigenvalue belongs to the smoothest sine mode, whose curvature is
$O(h^2)$; the largest belongs to the most oscillatory one.

**Solution.**

*Step 1.* The eigenvectors are $v^{(k)}_j = \sin\bigl(jk\pi/(N+1)\bigr)$ with eigenvalues

$$
\lambda_k = 2 - 2\cos\frac{k\pi}{N+1} = 4\sin^2\frac{k\pi}{2(N+1)} .
$$

*Step 2.* At $k = 1$, $\sin\theta \approx \theta$ gives
$\lambda_{\min} \approx \pi^2/(N+1)^2$.

*Step 3.* At $k = N$, $\sin\bigl(N\pi/(2(N+1))\bigr) \to 1$ gives $\lambda_{\max} \to 4$.

*Step 4.* Since $A$ is symmetric positive definite,
$\kappa_2 = \lambda_{\max}/\lambda_{\min} \approx 4(N+1)^2/\pi^2$.

$$
\boxed{\kappa_2(A) \approx \frac{4(N+1)^2}{\pi^2} = O(N^2) = O(h^{-2})}
$$

**Key takeaway.** Refining the mesh squares the difficulty. By Theorem 4.7 the CG iteration count
then grows like $N$, which Section 8.1 of the theory notebook measures on the two-dimensional
version.

In [17]:
print("   N    lambda_min     lambda_max     kappa_2      4(N+1)^2/pi^2")
for N in (8, 16, 32, 64):
    A_L = (np.diag(2.0 * np.ones(N)) + np.diag(-np.ones(N - 1), 1)
           + np.diag(-np.ones(N - 1), -1))
    lam = np.linalg.eigvalsh(A_L)
    closed = np.array([2 - 2 * np.cos(k * np.pi / (N + 1)) for k in range(1, N + 1)])
    assert np.allclose(np.sort(lam), np.sort(closed))
    print(f"  {N:3d}   {lam[0]:.6e}   {lam[-1]:.6f}   {lam[-1]/lam[0]:10.3f}   "
          f"{4*(N+1)**2/np.pi**2:10.3f}")
assert abs(np.linalg.cond(A_L) / (4 * (N + 1) ** 2 / np.pi ** 2) - 1) < 0.01

   N    lambda_min     lambda_max     kappa_2      4(N+1)^2/pi^2
    8   1.206148e-01   3.879385       32.163       32.828
   16   3.405380e-02   3.965946      116.461      117.127
   32   9.056155e-03   3.990944      440.689      441.355
   64   2.335546e-03   3.997664     1711.661     1712.328


### Problem L1.8 — Exact line search for steepest descent

**Statement.** For $f(x) = \tfrac12 x^{\top}Ax - b^{\top}x$ with $A$ symmetric positive
definite, minimize $f(x_k + \alpha r_k)$ over $\alpha$, where $r_k = b - Ax_k$.

**Intuition.** Restricted to a line, a positive definite quadratic is a parabola in $\alpha$,
and a parabola has one visible minimum.

**Solution.**

*Step 1.* Expand, using $Ax_k - b = -r_k$:

$$
\phi(\alpha) = f(x_k + \alpha r_k) = f(x_k) + \alpha \, r_k^{\top}(Ax_k - b) + \tfrac12 \alpha^2 r_k^{\top}Ar_k
= f(x_k) - \alpha \, r_k^{\top}r_k + \tfrac12\alpha^2 r_k^{\top}Ar_k .
$$

*Step 2.* $\phi'(\alpha) = -r_k^{\top}r_k + \alpha \, r_k^{\top}Ar_k$, and
$\phi''(\alpha) = r_k^{\top}Ar_k \gt 0$, so the stationary point is the minimum.

*Step 3.* Setting $\phi' = 0$,

$$
\alpha_k = \frac{r_k^{\top}r_k}{r_k^{\top}Ar_k} .
$$

*Step 4.* Consequently $r_{k+1}^{\top}r_k = r_k^{\top}r_k - \alpha_k r_k^{\top}Ar_k = 0$:
consecutive residuals are orthogonal, which is why steepest descent zig-zags.

$$
\boxed{\alpha_k = \frac{r_k^{\top}r_k}{r_k^{\top}Ar_k}, \qquad r_{k+1} \perp r_k}
$$

**Key takeaway.** Exact line search orthogonalizes only against the *previous* residual. CG's
$\alpha_k = r_k^{\top}r_k/(p_k^{\top}Ap_k)$ orthogonalizes against all of them at once, which is
the whole content of Theorem 4.6.

In [18]:
A_ls = np.array([[10.0, 1.0], [1.0, 1.0]])
b_ls = np.array([1.0, 2.0])
x = np.zeros(2)
x_ls = np.linalg.solve(A_ls, b_ls)


def energy(v):
    return np.sqrt(v @ A_ls @ v)


print("steepest descent with exact line search")
prev_e = energy(x_ls - x)
for k in range(6):
    r = b_ls - A_ls @ x
    alpha = (r @ r) / (r @ A_ls @ r)
    x = x + alpha * r
    r_new = b_ls - A_ls @ x
    e_now = energy(x_ls - x)
    print(f"  k={k}  alpha={alpha:.6f}  ||r||_2={np.linalg.norm(r_new):.4e}  "
          f"||e||_A={e_now:.4e}  r_k+1 . r_k = {r_new @ r:.2e}")
    assert abs(r_new @ r) < 1e-12
    assert e_now < prev_e
    prev_e = e_now
print("exact solution:", x_ls, " iterate:", x)

steepest descent with exact line search
  k=0  alpha=0.277778  ||r||_2=2.6087e+00  ||e||_A=1.6499e+00  r_k+1 . r_k = -4.44e-16
  k=1  alpha=0.135135  ||r||_2=1.4806e+00  ||e||_A=1.3426e+00  r_k+1 . r_k = -1.33e-15
  k=2  alpha=0.277778  ||r||_2=1.7274e+00  ||e||_A=1.0925e+00  r_k+1 . r_k = 0.00e+00
  k=3  alpha=0.135135  ||r||_2=9.8042e-01  ||e||_A=8.8901e-01  r_k+1 . r_k = 1.11e-16
  k=4  alpha=0.277778  ||r||_2=1.1438e+00  ||e||_A=7.2342e-01  r_k+1 . r_k = 1.11e-16
  k=5  alpha=0.135135  ||r||_2=6.4920e-01  ||e||_A=5.8867e-01  r_k+1 . r_k = -2.78e-16
exact solution: [-0.1111  2.1111]  iterate: [-0.0789  1.4982]


### Problem L1.9 — Conjugate directions decouple the minimization

**Statement.** Let $p_0, \dots, p_k$ be non-zero and $A$-conjugate for a symmetric positive
definite $A$. Show that $x_{k+1} = x_0 + \sum_{i=0}^{k}\alpha_i p_i$ with
$\alpha_i = p_i^{\top}r_0/(p_i^{\top}Ap_i)$ minimizes $f(x) = \tfrac12 x^{\top}Ax - b^{\top}x$
over $x_0 + \operatorname{span}\{p_0,\dots,p_k\}$.

**Intuition.** Conjugacy diagonalizes the quadratic form in the coordinates $\alpha_i$, so the
$k+1$-dimensional minimization splits into $k+1$ independent parabolas.

**Solution.**

*Step 1.* Write $x = x_0 + \sum_i \alpha_i p_i$ and expand $f$:

$$
f(x) = f(x_0) + \sum_{i} \alpha_i \, p_i^{\top}(Ax_0 - b) + \tfrac12 \sum_{i,j} \alpha_i \alpha_j \, p_i^{\top}Ap_j .
$$

*Step 2.* Conjugacy kills every cross term $i \neq j$, and $Ax_0 - b = -r_0$:

$$
f(x) = f(x_0) + \sum_{i} \left( -\alpha_i \, p_i^{\top}r_0 + \tfrac12 \alpha_i^2 \, p_i^{\top}Ap_i \right).
$$

*Step 3.* Each summand depends on one $\alpha_i$ only, so minimize term by term:
$\partial f/\partial \alpha_i = -p_i^{\top}r_0 + \alpha_i p_i^{\top}Ap_i = 0$.

*Step 4.* Positive definiteness gives $p_i^{\top}Ap_i \gt 0$, so each parabola is convex and the
stationary point is the minimum.

$$
\boxed{\alpha_i = \frac{p_i^{\top}r_0}{p_i^{\top}Ap_i}}
$$

**Key takeaway.** Any $A$-conjugate set solves the system in $n$ steps. CG's contribution is
producing such a set with a three-term recurrence instead of a full Gram-Schmidt.

In [19]:
m_cd = 5
Q_cd, _ = np.linalg.qr(rng.standard_normal((m_cd, m_cd)))
A_cd = Q_cd @ np.diag(np.linspace(1.0, 5.0, m_cd)) @ Q_cd.T
A_cd = (A_cd + A_cd.T) / 2
b_cd = rng.standard_normal(m_cd)
# build an A-conjugate set by Gram-Schmidt in the A-inner product
V = rng.standard_normal((m_cd, m_cd))
Pc = np.zeros((m_cd, m_cd))
for i in range(m_cd):
    v = V[:, i].copy()
    for j in range(i):
        v -= (Pc[:, j] @ A_cd @ v) / (Pc[:, j] @ A_cd @ Pc[:, j]) * Pc[:, j]
    Pc[:, i] = v
Gp = Pc.T @ A_cd @ Pc
print("max off-diagonal of P^T A P :", np.abs(Gp - np.diag(np.diag(Gp))).max())
x0 = np.zeros(m_cd)
r0 = b_cd - A_cd @ x0
x = x0 + sum((Pc[:, i] @ r0) / (Pc[:, i] @ A_cd @ Pc[:, i]) * Pc[:, i] for i in range(m_cd))
print("||x - A^-1 b|| :", np.linalg.norm(x - np.linalg.solve(A_cd, b_cd)))
assert np.abs(Gp - np.diag(np.diag(Gp))).max() < 1e-10
assert np.linalg.norm(x - np.linalg.solve(A_cd, b_cd)) < 1e-10

max off-diagonal of P^T A P : 1.2226743671544168e-15
||x - A^-1 b|| : 2.7755575615628914e-16


### Problem L1.10 — The CG recurrence coefficients

**Statement.** Derive $\alpha_k = r_k^{\top}r_k / (p_k^{\top}Ap_k)$ and
$\beta_k = r_{k+1}^{\top}r_{k+1}/(r_k^{\top}r_k)$ from the two design requirements
$r_{k+1} \perp p_k$ and $p_{k+1}^{\top}Ap_k = 0$.

**Intuition.** $\alpha_k$ is the exact line-search step along $p_k$; $\beta_k$ is the single
Gram-Schmidt coefficient needed to keep the new direction conjugate to the last one.

**Solution.**

*Step 1 — cite what may be used.* The identities $p_k^{\top}r_k = r_k^{\top}r_k$,
$r_{k+1}^{\top}r_k = 0$ and $p_k^{\top}Ap_{k-1} = 0$ are *not* free: they are statements (b),
(c) and (d) of the induction in Proof 5.5, established there simultaneously with the recurrences.
Deriving the coefficients from them is legitimate only in that order, and this problem assumes
Proof 5.5 up to index $k$.

*Step 2 — the step size.* From $r_{k+1} = r_k - \alpha_k A p_k$, imposing
$p_k^{\top}r_{k+1} = 0$ gives

$$
p_k^{\top}r_k - \alpha_k \, p_k^{\top}Ap_k = 0, \qquad \alpha_k = \frac{p_k^{\top}r_k}{p_k^{\top}Ap_k} .
$$

*Step 3.* By statement (d) of Proof 5.5, $p_k^{\top}r_k = r_k^{\top}r_k$, hence
$\alpha_k = r_k^{\top}r_k/(p_k^{\top}Ap_k)$.

*Step 4 — the conjugacy coefficient.* Impose $(r_{k+1} + \beta_k p_k)^{\top}Ap_k = 0$:

$$
\beta_k = -\frac{r_{k+1}^{\top}Ap_k}{p_k^{\top}Ap_k} .
$$

*Step 5.* Rearranging the residual recurrence, $Ap_k = (r_k - r_{k+1})/\alpha_k$, so with
$r_{k+1}^{\top}r_k = 0$ from Proof 5.5 Step 1,

$$
r_{k+1}^{\top}Ap_k = \frac{r_{k+1}^{\top}r_k - r_{k+1}^{\top}r_{k+1}}{\alpha_k} = -\frac{r_{k+1}^{\top}r_{k+1}}{\alpha_k} .
$$

*Step 6.* Substituting and using $\alpha_k p_k^{\top}Ap_k = r_k^{\top}r_k$,

$$
\beta_k = \frac{r_{k+1}^{\top}r_{k+1}}{\alpha_k \, p_k^{\top}Ap_k} = \frac{r_{k+1}^{\top}r_{k+1}}{r_k^{\top}r_k} .
$$

$$
\boxed{\alpha_k = \frac{r_k^{\top}r_k}{p_k^{\top}Ap_k}, \qquad \beta_k = \frac{r_{k+1}^{\top}r_{k+1}}{r_k^{\top}r_k}}
$$

**Key takeaway.** Both coefficients cost one inner product each, and neither refers to any vector
older than step $k$. That is the payoff of the simultaneous induction in Proof 5.5.

In [20]:
m_c = 12
Q_c, _ = np.linalg.qr(rng.standard_normal((m_c, m_c)))
A_c = Q_c @ np.diag(np.linspace(1.0, 8.0, m_c)) @ Q_c.T
A_c = (A_c + A_c.T) / 2
b_c = rng.standard_normal(m_c)
x = np.zeros(m_c)
r = b_c - A_c @ x
p = r.copy()
print(" k   alpha (r.r/pAp)   alpha (p.r/pAp)   beta (new/old)   -r'Ap/pAp")
for k in range(5):
    Ap = A_c @ p
    a1 = (r @ r) / (p @ Ap)
    a2 = (p @ r) / (p @ Ap)
    x = x + a1 * p
    r_new = r - a1 * Ap
    b1 = (r_new @ r_new) / (r @ r)
    b2 = -(r_new @ Ap) / (p @ Ap)
    print(f" {k}   {a1:.12f}    {a2:.12f}    {b1:.12f}   {b2:.12f}")
    assert abs(a1 - a2) < 1e-12 and abs(b1 - b2) < 1e-12
    r, p = r_new, r_new + b1 * p

 k   alpha (r.r/pAp)   alpha (p.r/pAp)   beta (new/old)   -r'Ap/pAp
 0   0.289465887658    0.289465887658    0.262117047232   0.262117047232
 1   0.266915720564    0.266915720564    0.367720873635   0.367720873635
 2   0.369223360303    0.369223360303    0.508310816529   0.508310816529
 3   0.243723398320    0.243723398320    0.083506019406   0.083506019406
 4   0.214606690667    0.214606690667    0.081848949937   0.081848949937


### Problem L1.11 — The Arnoldi relation

**Statement.** Derive $AQ_k = Q_{k+1}\bar{H}_k$ from the Arnoldi recurrence of Definition 3.9,
and say why $\bar{H}_k$ is upper Hessenberg.

**Intuition.** Arnoldi is Gram-Schmidt applied to $v, Av, A^2v, \dots$; the coefficient array of
a Gram-Schmidt run on a Krylov sequence is Hessenberg because $Aq_j$ only reaches one step
further than $q_j$.

**Solution.**

*Step 1.* Step $j$ computes $h_{ij} = q_i^{\top}Aq_j$ for $i \le j$, subtracts, and normalizes the
remainder:

$$
v_j = Aq_j - \sum_{i=1}^{j} h_{ij}q_i, \qquad h_{j+1,j} = \lVert v_j \rVert_2, \qquad q_{j+1} = v_j/h_{j+1,j} .
$$

*Step 2.* Rearranging, $Aq_j = \sum_{i=1}^{j+1} h_{ij}q_i$.

*Step 3.* That is column $j$ of $AQ_k = Q_{k+1}\bar{H}_k$, where $\bar{H}_k$ has entries $h_{ij}$
and $h_{ij} = 0$ for $i \gt j+1$ — nothing beyond $q_{j+1}$ is ever produced.

*Step 4.* A matrix with $h_{ij} = 0$ for $i \gt j+1$ is upper Hessenberg by definition.

$$
\boxed{AQ_k = Q_k H_k + h_{k+1,k}q_{k+1}e_k^{\top} = Q_{k+1}\bar{H}_k}
$$

**Key takeaway.** The relation is the entire computational content of GMRES: it moves an
$n$-dimensional problem into a $(k+1) \times k$ one, which is Theorem 4.8 part 2.

In [21]:
m_a = 8
A_a = rng.standard_normal((m_a, m_a))
v0 = rng.standard_normal(m_a)
k_a = 5
Qa = np.zeros((m_a, k_a + 1))
H = np.zeros((k_a + 1, k_a))
Qa[:, 0] = v0 / np.linalg.norm(v0)
for j in range(k_a):
    w = A_a @ Qa[:, j]
    for i in range(j + 1):
        H[i, j] = Qa[:, i] @ w
        w = w - H[i, j] * Qa[:, i]
    H[j + 1, j] = np.linalg.norm(w)
    Qa[:, j + 1] = w / H[j + 1, j]
print("||A Q_k - Q_{k+1} H_bar||_F :", np.linalg.norm(A_a @ Qa[:, :k_a] - Qa @ H))
print("||Q^T Q - I||_F             :", np.linalg.norm(Qa.T @ Qa - np.eye(k_a + 1)))
print("entries below the subdiagonal:", np.abs(np.tril(H, -2)).max())
assert np.linalg.norm(A_a @ Qa[:, :k_a] - Qa @ H) < 1e-12
assert np.abs(np.tril(H, -2)).max() == 0.0

||A Q_k - Q_{k+1} H_bar||_F : 1.233244451751024e-15
||Q^T Q - I||_F             : 1.1137075018563528e-15
entries below the subdiagonal: 0.0


### Problem L1.12 — GMRES as a small least-squares problem

**Statement.** Show that minimizing $\lVert b - Ax \rVert_2$ over $x \in x_0 + \mathcal{K}_k(A,r_0)$
is the same as minimizing $\lVert \beta e_1 - \bar{H}_k y \rVert_2$ over $y \in \mathbb{R}^k$,
where $\beta = \lVert r_0 \rVert_2$.

**Intuition.** An orthonormal basis is a rigid change of coordinates: it moves the problem into
$k+1$ dimensions without changing any length.

**Solution.**

*Step 1.* Set $q_1 = r_0/\beta$, so $r_0 = \beta q_1 = Q_{k+1}(\beta e_1)$.

*Step 2.* For $x = x_0 + Q_k y$,
$b - Ax = r_0 - AQ_k y$.

*Step 3.* Apply the Arnoldi relation of Problem L1.11:

$$
b - Ax = Q_{k+1}(\beta e_1) - Q_{k+1}\bar{H}_k y = Q_{k+1}\bigl(\beta e_1 - \bar{H}_k y\bigr).
$$

*Step 4.* $Q_{k+1}$ has orthonormal columns, so $\lVert Q_{k+1}z \rVert_2 = \lVert z \rVert_2$
for every $z$, and the two minimizations coincide.

$$
\boxed{\lVert b - Ax_k \rVert_2 = \min_{y} \lVert \beta e_1 - \bar{H}_k y \rVert_2}
$$

**Key takeaway.** The small problem is solved by Givens rotations in $O(k)$ work per step,
because $\bar{H}_k$ gains one column and one subdiagonal entry at a time.

In [22]:
b_ls2 = rng.standard_normal(m_a)
beta = np.linalg.norm(b_ls2)
Qg = np.zeros((m_a, k_a + 1))
Hg = np.zeros((k_a + 1, k_a))
Qg[:, 0] = b_ls2 / beta
for j in range(k_a):
    w = A_a @ Qg[:, j]
    for i in range(j + 1):
        Hg[i, j] = Qg[:, i] @ w
        w = w - Hg[i, j] * Qg[:, i]
    Hg[j + 1, j] = np.linalg.norm(w)
    Qg[:, j + 1] = w / Hg[j + 1, j]
rhs = np.zeros(k_a + 1)
rhs[0] = beta
y, *_ = np.linalg.lstsq(Hg, rhs, rcond=None)
x_small = Qg[:, :k_a] @ y
res_small = np.linalg.norm(rhs - Hg @ y)
res_full = np.linalg.norm(b_ls2 - A_a @ x_small)
# brute-force minimization over the same Krylov space
Kb = np.column_stack([np.linalg.matrix_power(A_a, j) @ b_ls2 for j in range(k_a)])
z, *_ = np.linalg.lstsq(A_a @ Kb, b_ls2, rcond=None)
res_brute = np.linalg.norm(b_ls2 - A_a @ (Kb @ z))
print(f"small least-squares residual : {res_small:.12f}")
print(f"true residual of that iterate: {res_full:.12f}")
print(f"brute-force Krylov minimum   : {res_brute:.12f}")
assert abs(res_small - res_full) < 1e-10 and abs(res_small - res_brute) < 1e-8

small least-squares residual : 0.981704052914
true residual of that iterate: 0.981704052914
brute-force Krylov minimum   : 0.981704052914


### Problem L1.13 — GMRES residuals never increase

**Statement.** Prove $\lVert r_{k+1} \rVert_2 \le \lVert r_k \rVert_2$ for GMRES.

**Intuition.** Each step minimizes over a strictly larger set, and enlarging a feasible set can
only lower a minimum.

**Solution.**

*Step 1.* By definition
$\lVert r_k \rVert_2 = \min_{x \in x_0 + \mathcal{K}_k} \lVert b - Ax \rVert_2$.

*Step 2.* The Krylov spaces are nested: $\mathcal{K}_k \subseteq \mathcal{K}_{k+1}$, since every
generator of $\mathcal{K}_k$ is a generator of $\mathcal{K}_{k+1}$.

*Step 3.* Hence $x_0 + \mathcal{K}_k \subseteq x_0 + \mathcal{K}_{k+1}$, and the minimum over the
larger affine set is no larger.

$$
\boxed{\lVert r_{k+1} \rVert_2 \le \lVert r_k \rVert_2}
$$

**Key takeaway.** Monotone does not mean *strictly* decreasing: Example 6.4 of the theory
notebook holds the residual at exactly $1$ for $n-1$ steps. This is Theorem 4.8 part 3.

In [23]:
A_m = rng.standard_normal((30, 30)) + 3.0 * np.eye(30)
b_m = rng.standard_normal(30)
beta = np.linalg.norm(b_m)
Qm = np.zeros((30, 21))
Hm = np.zeros((21, 20))
Qm[:, 0] = b_m / beta
res_hist = [beta]
for j in range(20):
    w = A_m @ Qm[:, j]
    for i in range(j + 1):
        Hm[i, j] = Qm[:, i] @ w
        w = w - Hm[i, j] * Qm[:, i]
    Hm[j + 1, j] = np.linalg.norm(w)
    if Hm[j + 1, j] > 1e-14:
        Qm[:, j + 1] = w / Hm[j + 1, j]
    rr = np.zeros(j + 2)
    rr[0] = beta
    yy, *_ = np.linalg.lstsq(Hm[:j + 2, :j + 1], rr, rcond=None)
    res_hist.append(np.linalg.norm(rr - Hm[:j + 2, :j + 1] @ yy))
res_hist = np.array(res_hist)
print("relative residuals:", np.round(res_hist / res_hist[0], 8))
print("non-increasing    :", bool(np.all(np.diff(res_hist) <= 1e-12)))
assert np.all(np.diff(res_hist) <= 1e-12)

relative residuals: [1.     0.9047 0.8803 0.854  0.8444 0.8436 0.8401 0.8346 0.8119 0.8119
 0.6944 0.693  0.6785 0.6783 0.6727 0.6708 0.6619 0.6558 0.6422 0.61
 0.5919]
non-increasing    : True


### Problem L1.14 — Lanczos: Arnoldi for a symmetric matrix

**Statement.** Show that if $A = A^{\top}$ then the Arnoldi Hessenberg matrix $H_k = Q_k^{\top}AQ_k$
is symmetric tridiagonal, and deduce the three-term Lanczos recurrence.

**Intuition.** A matrix cannot be both symmetric and one-sided; forcing both leaves only the
three central diagonals.

**Solution.**

*Step 1.* $H_k = Q_k^{\top}AQ_k$, so
$H_k^{\top} = Q_k^{\top}A^{\top}Q_k = Q_k^{\top}AQ_k = H_k$: symmetric.

*Step 2.* Arnoldi gives $h_{ij} = 0$ for $i \gt j+1$: upper Hessenberg.

*Step 3.* Symmetry transfers those zeros above the diagonal: $h_{ij} = h_{ji} = 0$ for
$j \gt i + 1$. So $h_{ij} = 0$ whenever $\lvert i - j \rvert \gt 1$, that is, $H_k$ is
tridiagonal, written $T_k$ with diagonal $\alpha_j$ and off-diagonal $\beta_j$.

*Step 4.* Column $j$ of $AQ_k = Q_{k+1}\bar{T}_k$ therefore reads

$$
\beta_j q_{j+1} = Aq_j - \alpha_j q_j - \beta_{j-1}q_{j-1} ,
$$

a three-term recurrence: only two previous vectors are needed.

$$
\boxed{H_k = T_k = \operatorname{tridiag}(\beta_{j-1}, \alpha_j, \beta_j)}
$$

**Key takeaway.** Lanczos is why CG needs three vectors and GMRES needs $k$. The two are the same
algorithm applied to a symmetric and a non-symmetric matrix respectively.

In [24]:
m_l = 10
Ql, _ = np.linalg.qr(rng.standard_normal((m_l, m_l)))
A_l = Ql @ np.diag(np.linspace(1.0, 6.0, m_l)) @ Ql.T
A_l = (A_l + A_l.T) / 2
v_l = rng.standard_normal(m_l)
k_l = 6
Qz = np.zeros((m_l, k_l + 1))
Hz = np.zeros((k_l + 1, k_l))
Qz[:, 0] = v_l / np.linalg.norm(v_l)
for j in range(k_l):
    w = A_l @ Qz[:, j]
    for i in range(j + 1):
        Hz[i, j] = Qz[:, i] @ w
        w = w - Hz[i, j] * Qz[:, i]
    Hz[j + 1, j] = np.linalg.norm(w)
    Qz[:, j + 1] = w / Hz[j + 1, j]
Hk = Hz[:k_l, :k_l]
print("H_k (rounded):\n", np.round(Hk, 6))
print("||H_k - H_k^T||_F                 :", np.linalg.norm(Hk - Hk.T))
print("mass outside the three diagonals  :",
      np.abs(Hk - np.triu(np.tril(Hk, 1), -1)).max())
assert np.linalg.norm(Hk - Hk.T) < 1e-10
assert np.abs(Hk - np.triu(np.tril(Hk, 1), -1)).max() < 1e-10

H_k (rounded):
 [[ 3.1961  1.2373 -0.      0.     -0.      0.    ]
 [ 1.2373  3.3603  1.3892  0.      0.     -0.    ]
 [ 0.      1.3892  3.1551  1.3841  0.     -0.    ]
 [ 0.      0.      1.3841  3.5884  1.2923 -0.    ]
 [ 0.      0.      0.      1.2923  3.7904  0.9814]
 [ 0.      0.      0.      0.      0.9814  4.7816]]
||H_k - H_k^T||_F                 : 2.1603824672750337e-14
mass outside the three diagonals  : 9.908740494779522e-15


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Gradient descent is Richardson iteration

**Statement.** Show that gradient descent with fixed step $\eta$ on
$f(x) = \tfrac12 x^{\top}Ax - b^{\top}x$, with $A$ symmetric positive definite, is a stationary
iteration, and identify $G$ and the splitting.

**Intuition.** The gradient of a quadratic is the negative residual, so a gradient step is a
scaled residual correction — exactly what a stationary method does.

**Solution.**

*Step 1.* $\nabla f(x) = Ax - b = -r(x)$.

*Step 2.* The update is $x_{k+1} = x_k - \eta(Ax_k - b) = (I - \eta A)x_k + \eta b$.

*Step 3.* Matching Definition 3.4, $G = I - \eta A = M^{-1}N$ with $M = \eta^{-1}I$ and
$N = \eta^{-1}I - A$.

$$
\boxed{G_{\mathrm{GD}} = I - \eta A, \qquad M = \eta^{-1}I}
$$

**Key takeaway.** Every convergence statement about learning rates on a quadratic loss is a
statement about $\rho(I - \eta A)$, so the entire theory of Section 4 transfers to optimization.

In [25]:
m_gd = 5
Q_gd, _ = np.linalg.qr(rng.standard_normal((m_gd, m_gd)))
A_gd = Q_gd @ np.diag(np.linspace(0.5, 4.0, m_gd)) @ Q_gd.T
A_gd = (A_gd + A_gd.T) / 2
b_gd = rng.standard_normal(m_gd)
eta = 0.3
x1 = np.zeros(m_gd)
x2 = np.zeros(m_gd)
G_gd = np.eye(m_gd) - eta * A_gd
for _ in range(50):
    x1 = x1 - eta * (A_gd @ x1 - b_gd)
    x2 = G_gd @ x2 + eta * b_gd
print("gradient descent iterate :", x1)
print("stationary iterate       :", x2)
print("rho(I - eta A)           :", max(abs(np.linalg.eigvals(G_gd))))
assert np.allclose(x1, x2)

gradient descent iterate : [ 0.1671 -0.1103 -0.012  -0.8154  0.8046]
stationary iterate       : [ 0.1671 -0.1103 -0.012  -0.8154  0.8046]
rho(I - eta A)           : 0.8499999999999991


### Problem L2.2 — The optimal fixed step size

**Statement.** For symmetric positive definite $A$ with $0 \lt \lambda_{\min} \le \lambda_{\max}$,
find the $\eta$ minimizing $\rho(I - \eta A)$ and the resulting rate.

**Intuition.** The two extreme eigenvalues pull in opposite directions; the best step balances
them so that neither dominates.

**Solution.**

*Step 1.* The eigenvalues of $I - \eta A$ are $1 - \eta\lambda_i$, so

$$
\rho(I-\eta A) = \max\bigl( \lvert 1 - \eta\lambda_{\min} \rvert, \ \lvert 1-\eta\lambda_{\max} \rvert \bigr).
$$

*Step 2.* The first term increases with $\eta$ in modulus only after $\eta \gt 2/\lambda_{\min}$;
the second decreases then increases. The maximum of two such functions is minimized where they
are equal with opposite signs:

$$
1 - \eta\lambda_{\min} = -(1 - \eta\lambda_{\max}).
$$

*Step 3.* Solving, $\eta^{\star} = 2/(\lambda_{\min} + \lambda_{\max})$.

*Step 4.* Substituting back,

$$
\rho^{\star} = 1 - \frac{2\lambda_{\min}}{\lambda_{\min}+\lambda_{\max}} = \frac{\lambda_{\max}-\lambda_{\min}}{\lambda_{\max}+\lambda_{\min}} = \frac{\kappa - 1}{\kappa+1}.
$$

$$
\boxed{\eta^{\star} = \frac{2}{\lambda_{\min}+\lambda_{\max}}, \qquad \rho^{\star} = \frac{\kappa-1}{\kappa+1}}
$$

**Key takeaway.** Gradient descent pays $\kappa$ where CG pays $\sqrt{\kappa}$ (Theorem 4.7); at
$\kappa = 10^4$ that is the difference between $0.9998$ and $0.9802$ per step.

In [26]:
lam_min, lam_max = 0.5, 4.0
eta_star = 2 / (lam_min + lam_max)
grid = np.linspace(0.01, 2 / lam_max * 1.5, 20001)
rho_grid = np.maximum(abs(1 - grid * lam_min), abs(1 - grid * lam_max))
print(f"eta*  formula   = {eta_star:.6f}   grid minimizer = {grid[rho_grid.argmin()]:.6f}")
print(f"rho*  formula   = {(lam_max-lam_min)/(lam_max+lam_min):.6f}   grid minimum   = {rho_grid.min():.6f}")
for kap in (10.0, 100.0, 1e4):
    print(f"  kappa = {kap:8.0f}:  GD rate {(kap-1)/(kap+1):.6f}   "
          f"CG rate {(np.sqrt(kap)-1)/(np.sqrt(kap)+1):.6f}")
assert abs(grid[rho_grid.argmin()] - eta_star) < 1e-3
assert abs(rho_grid.min() - (lam_max - lam_min) / (lam_max + lam_min)) < 1e-4

eta*  formula   = 0.444444   grid minimizer = 0.444417
rho*  formula   = 0.777778   grid minimum   = 0.777791
  kappa =       10:  GD rate 0.818182   CG rate 0.519494
  kappa =      100:  GD rate 0.980198   CG rate 0.818182
  kappa =    10000:  GD rate 0.999800   CG rate 0.980198


### Problem L2.3 — How many CG iterations for six digits

**Statement.** For a symmetric positive definite $A$ with $\kappa_2(A) = 100$, how many CG
iterations guarantee $\lVert e_k \rVert_A \le 10^{-6}\lVert e_0 \rVert_A$?

**Intuition.** Invert the geometric bound of Theorem 4.7; only the square root of $\kappa$ enters.

**Solution.**

*Step 1.* Theorem 4.7 gives $\lVert e_k \rVert_A \le 2\gamma^{k}\lVert e_0 \rVert_A$ with
$\gamma = (\sqrt\kappa-1)/(\sqrt\kappa+1)$.

*Step 2.* $\sqrt{100} = 10$, so $\gamma = 9/11 = 0.818182$.

*Step 3.* Require $2\gamma^k \le 10^{-6}$, that is $\gamma^k \le 5 \times 10^{-7}$.

*Step 4.* Taking logarithms, $k \ge \ln(5 \times 10^{-7}) / \ln(9/11) = 14.5086/0.200671 = 72.30$.

*Step 5.* Round up: $k = 73$.

$$
\boxed{k = 73 \text{ iterations}}
$$

**Key takeaway.** Gradient descent on the same system contracts at $\rho = 99/101$ and needs
$691$ iterations for the same reduction — the square root is worth an order of magnitude.

In [27]:
kap = 100.0
gam = (np.sqrt(kap) - 1) / (np.sqrt(kap) + 1)
k_cg = int(np.ceil(np.log(1e-6 / 2) / np.log(gam)))
rho_gd = (kap - 1) / (kap + 1)
k_gd = int(np.ceil(np.log(1e-6) / np.log(rho_gd)))
print(f"gamma = {gam:.6f}  = 9/11 = {9/11:.6f}")
print(f"exact threshold  k >= {np.log(1e-6/2)/np.log(gam):.4f}  ->  k = {k_cg}")
print(f"gradient descent rho = {rho_gd:.6f}  ->  k = {k_gd}")
assert k_cg == 73

gamma = 0.818182  = 9/11 = 0.818182
exact threshold  k >= 72.3008  ->  k = 73
gradient descent rho = 0.980198  ->  k = 691


### Problem L2.4 — Symmetric preconditioning and PCG

**Statement.** Let $M = LL^{\top}$ be a symmetric positive definite preconditioner. Show how to
turn $Ax = b$ into a symmetric positive definite system whose condition number is
$\kappa_2(M^{-1}A)$.

**Intuition.** Left-multiplying by $M^{-1}$ destroys symmetry; splitting $M$ in half and
applying one half to each side preserves it.

**Solution.**

*Step 1.* Insert the identity: $A L^{-\top}L^{\top}x = b$.

*Step 2.* Multiply on the left by $L^{-1}$:
$(L^{-1}AL^{-\top})(L^{\top}x) = L^{-1}b$, that is $\tilde{A}\tilde{x} = \tilde{b}$.

*Step 3.* $\tilde{A}^{\top} = L^{-1}A^{\top}L^{-\top} = \tilde{A}$ and
$v^{\top}\tilde{A}v = (L^{-\top}v)^{\top}A(L^{-\top}v) \gt 0$: symmetric positive definite.

*Step 4.* $L^{-\top}\tilde{A}L^{\top} = M^{-1}A$, so the two matrices are similar and share a
spectrum, hence $\kappa_2(\tilde A) = \kappa_2(M^{-1}A)$.

$$
\boxed{\tilde{A} = L^{-1}AL^{-\top}, \qquad \kappa_2(\tilde{A}) = \kappa_2(M^{-1}A)}
$$

**Key takeaway.** This is Theorem 4.9. In practice $L$ is never formed: rewriting the recurrences
in the original variables turns every appearance of $L$ into one solve with $M$.

In [28]:
m_p = 30
Qp, _ = np.linalg.qr(rng.standard_normal((m_p, m_p)))
A_p = Qp @ np.diag(np.logspace(0, 3, m_p)) @ Qp.T
A_p = (A_p + A_p.T) / 2
Mp = np.diag(np.diag(A_p))
Lp = np.linalg.cholesky(Mp)
At = np.linalg.solve(Lp, np.linalg.solve(Lp, A_p.T).T)
print("||At - At^T||_F        :", np.linalg.norm(At - At.T))
print("min eigenvalue of At   :", np.linalg.eigvalsh(At).min())
lam_MA = np.sort(np.linalg.eigvals(np.linalg.solve(Mp, A_p)).real)
print("spec(At) vs spec(M^-1 A) max difference:",
      np.abs(lam_MA - np.sort(np.linalg.eigvalsh(At))).max())
print(f"kappa_2(A)       = {np.linalg.cond(A_p):.2f}")
print(f"kappa_2(At)      = {np.linalg.cond(At):.2f}"
      f"   eigenvalue ratio of M^-1 A = {lam_MA[-1]/lam_MA[0]:.2f}")
assert np.linalg.norm(At - At.T) < 1e-9
assert abs(np.linalg.cond(At) - lam_MA[-1] / lam_MA[0]) < 1e-6

||At - At^T||_F        : 7.957163573927999e-16
min eigenvalue of At   : 0.007178442106026453
spec(At) vs spec(M^-1 A) max difference: 1.687538997430238e-14
kappa_2(A)       = 1000.00
kappa_2(At)      = 770.96   eigenvalue ratio of M^-1 A = 770.96


### Problem L2.5 — A preconditioner that clusters the spectrum

**Statement.** If $M$ is chosen so that $\operatorname{spec}(M^{-1}A) \subseteq [1-\epsilon, 1+\epsilon]$
with $0 \lt \epsilon \lt 1$, show the preconditioned Richardson iteration
$x_{k+1} = x_k + M^{-1}(b - Ax_k)$ contracts by at least $\epsilon$ per step.

**Intuition.** A perfect preconditioner makes $M^{-1}A = I$, so the iteration converges in one
step; $\epsilon$ measures how far from perfect it is.

**Solution.**

*Step 1.* Rearranging, $x_{k+1} = (I - M^{-1}A)x_k + M^{-1}b$, so $G = I - M^{-1}A$.

*Step 2.* If $\lambda$ is an eigenvalue of $M^{-1}A$, then $1 - \lambda$ is one of $G$.

*Step 3.* $\lambda \in [1-\epsilon, 1+\epsilon]$ gives $1 - \lambda \in [-\epsilon, \epsilon]$,
hence $\rho(G) \le \epsilon \lt 1$.

*Step 4.* Theorem 4.2 then gives convergence with asymptotic factor at most $\epsilon$.

$$
\boxed{\rho(I - M^{-1}A) \le \epsilon}
$$

**Key takeaway.** Preconditioning is spectral surgery, not scaling: what matters is where the
eigenvalues of $M^{-1}A$ sit, which is why Section 7.2's clustered spectrum converged four times
faster than the spread one at identical $\kappa$.

In [29]:
m_e = 40
Qe, _ = np.linalg.qr(rng.standard_normal((m_e, m_e)))
eps_cl = 0.2
spec_cl = 1.0 + eps_cl * np.linspace(-1.0, 1.0, m_e)
A_e = Qe @ np.diag(spec_cl) @ Qe.T
A_e = (A_e + A_e.T) / 2
b_e = rng.standard_normal(m_e)
x_e = np.zeros(m_e)
x_true = np.linalg.solve(A_e, b_e)
err = [np.linalg.norm(x_e - x_true)]
for _ in range(30):
    x_e = x_e + (b_e - A_e @ x_e)
    err.append(np.linalg.norm(x_e - x_true))
err = np.array(err)
print(f"spectrum in [{spec_cl.min():.2f}, {spec_cl.max():.2f}],  epsilon = {eps_cl}")
print(f"rho(I - A) = {max(abs(np.linalg.eigvals(np.eye(m_e) - A_e))):.6f}")
print(f"observed contraction over the first 12 steps: {(err[12]/err[0])**(1/12):.6f}")
print(f"relative error after 12 steps : {err[12]/err[0]:.3e}   epsilon^12 = {eps_cl**12:.3e}")
print(f"relative error after 30 steps : {err[30]/err[0]:.3e}   (round-off floor reached)")
assert (err[12] / err[0]) ** (1 / 12) <= eps_cl + 1e-9

spectrum in [0.80, 1.20],  epsilon = 0.2
rho(I - A) = 0.200000
observed contraction over the first 12 steps: 0.175529
relative error after 12 steps : 8.554e-10   epsilon^12 = 4.096e-09
relative error after 30 steps : 1.866e-16   (round-off floor reached)


### Problem L2.6 — MINRES for symmetric indefinite systems

**Statement.** Explain how MINRES minimizes $\lVert b - Ax_k \rVert_2$ over
$x_0 + \mathcal{K}_k(A, r_0)$ for symmetric indefinite $A$ using $O(1)$ vectors of storage per
step, and say why CG cannot be used instead.

**Intuition.** GMRES is optimal but stores everything; Lanczos makes the Hessenberg matrix
tridiagonal, and a tridiagonal least-squares problem can be updated with short recurrences.

**Solution.**

*Step 1.* For symmetric $A$, Problem L1.14 gives $AQ_k = Q_{k+1}\bar{T}_k$ with $\bar{T}_k$
tridiagonal plus one extra subdiagonal entry.

*Step 2.* By Problem L1.12 the residual minimization is
$\min_y \lVert \beta e_1 - \bar{T}_k y \rVert_2$.

*Step 3.* $\bar{T}_k$ has bandwidth $3$, so its QR factorization by Givens rotations touches only
the last two rows at each step, and $R_k$ has upper bandwidth $2$.

*Step 4.* The solution can therefore be updated as $x_k = x_{k-1} + \tau_k d_k$ with $d_k$ from a
three-term recurrence: $O(1)$ vectors and $O(n)$ flops per step.

*Step 5.* CG is unavailable because $\lVert \cdot \rVert_A$ is not a norm when $A$ is indefinite;
Section 7.4 of the theory notebook shows $p_0^{\top}Ap_0 = 0$ breaking the very first step.

$$
\boxed{\text{MINRES: GMRES optimality with Lanczos storage, valid for symmetric indefinite } A}
$$

**Key takeaway.** Symmetry buys short recurrences; positive definiteness buys the energy norm.
MINRES keeps the first and gives up the second.

In [30]:
from scipy.sparse.linalg import minres, cg as scipy_cg

m_i = 60
Qi, _ = np.linalg.qr(rng.standard_normal((m_i, m_i)))
spec_i = np.concatenate([np.linspace(-2.0, -0.5, m_i // 2), np.linspace(0.5, 2.0, m_i // 2)])
A_i = Qi @ np.diag(spec_i) @ Qi.T
A_i = (A_i + A_i.T) / 2
b_i = rng.standard_normal(m_i)
x_exact = np.linalg.solve(A_i, b_i)
x_mr, info_mr = minres(A_i, b_i, rtol=1e-10, maxiter=500)
print("A is symmetric   :", np.allclose(A_i, A_i.T))
print("A is indefinite  : eigenvalues from", spec_i.min(), "to", spec_i.max())
print(f"MINRES relative error : {np.linalg.norm(x_mr - x_exact)/np.linalg.norm(x_exact):.3e}"
      f"   info = {info_mr}")
with np.errstate(divide="ignore", invalid="ignore"):
    x_cg, info_cg = scipy_cg(A_i, b_i, rtol=1e-10, atol=0.0, maxiter=500)
    err_cg = np.linalg.norm(x_cg - x_exact) / np.linalg.norm(x_exact)
print(f"CG on the same system : relative error {err_cg:.3e}   info = {info_cg}"
      "   (no guarantee: A is indefinite)")
# the guarantee really is absent: here is an exact breakdown
A_brk = np.diag([1.0, -1.0])
p_brk = np.array([1.0, 1.0])
print(f"CG on diag(1, -1) with b = (1,1): p_0^T A p_0 = {p_brk @ A_brk @ p_brk:.1f}"
      "  -> division by zero at step 0")
assert np.linalg.norm(x_mr - x_exact) / np.linalg.norm(x_exact) < 1e-6
assert p_brk @ A_brk @ p_brk == 0.0

A is symmetric   : True
A is indefinite  : eigenvalues from -2.0 to 2.0
MINRES relative error : 6.375e-10   info = 0
CG on the same system : relative error 7.345e-12   info = 0   (no guarantee: A is indefinite)
CG on diag(1, -1) with b = (1,1): p_0^T A p_0 = 0.0  -> division by zero at step 0


### Problem L2.7 — Electrostatic potential of a uniformly charged slab (physics)

**Statement.** A slab occupies $0 \le x \le d$ with uniform charge density $\rho_0$ and grounded
faces, $\varphi(0) = \varphi(d) = 0$. Poisson's equation is
$-\varphi''(x) = \rho_0/\varepsilon_0$. Discretize with $N$ interior points, solve by CG, and
compare with the exact potential. Give the peak potential.

**Intuition.** The second difference of a quadratic is exact, so the three-point Laplacian
reproduces the parabolic potential of a uniform slab with no discretization error at all.

**Solution.**

*Step 1 — exact solution.* Integrating twice with $\varphi(0)=\varphi(d)=0$,

$$
\varphi(x) = \frac{\rho_0}{2\varepsilon_0}\, x(d-x), \qquad
\varphi_{\max} = \varphi(d/2) = \frac{\rho_0 d^2}{8\varepsilon_0}.
$$

*Step 2 — discretize.* With $h = d/(N+1)$ and $x_j = jh$, the second difference gives

$$
\frac{-\varphi_{j-1} + 2\varphi_j - \varphi_{j+1}}{h^{2}} = \frac{\rho_0}{\varepsilon_0},
\qquad \text{that is} \qquad A\varphi = \frac{h^{2}\rho_0}{\varepsilon_0}\mathbf{1},
$$

with $A = \operatorname{tridiag}(-1,2,-1)$, which is symmetric positive definite — so CG applies.

*Step 3 — no discretization error.* The exact solution is a quadratic, and the second difference
of a quadratic equals its second derivative exactly. Hence the discrete solution equals the exact
one at every grid point.

*Step 4 — cost.* By Problem L1.7, $\kappa_2(A) \approx 4(N+1)^2/\pi^2$, so by Theorem 4.7 the CG
iteration count grows like $\sqrt{\kappa_2} \approx 2(N+1)/\pi$, that is, linearly in $N$.

$$
\boxed{\varphi(x) = \frac{\rho_0}{2\varepsilon_0}x(d-x), \qquad \varphi_{\max} = \frac{\rho_0 d^{2}}{8\varepsilon_0}}
$$

**Key takeaway.** The discretization error is exactly zero, so the run isolates the solver. The
measured count is $(N+1)/2$ — the constant source excites only the modes symmetric about the
midpoint, so Theorem 4.6 statement 5 terminates CG on half the spectrum — and it doubles when $N$
doubles, exactly as $\sqrt{\kappa_2}$ does.

In [31]:
def cg_count(A, b, tol=1e-12, maxit=5000):
    x = np.zeros(len(b))
    r = b - A @ x
    p = r.copy()
    rr = r @ r
    nb = np.linalg.norm(b)
    it = 0
    while np.sqrt(rr) > tol * nb and it < maxit:
        Ap = A @ p
        alpha = rr / (p @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        rr_new = r @ r
        p = r + (rr_new / rr) * p
        rr = rr_new
        it += 1
    return x, it


rho0, eps0, d_slab = 1.0, 1.0, 1.0
print("   N       h        max|phi_num - phi_exact|   phi_max      CG iters   sqrt(kappa)")
for N in (15, 31, 63, 127):
    h = d_slab / (N + 1)
    A_ph = (np.diag(2.0 * np.ones(N)) + np.diag(-np.ones(N - 1), 1)
            + np.diag(-np.ones(N - 1), -1))
    rhs = h ** 2 * rho0 / eps0 * np.ones(N)
    phi, it = cg_count(A_ph, rhs)
    xs = np.arange(1, N + 1) * h
    phi_exact = rho0 / (2 * eps0) * xs * (d_slab - xs)
    kappa_ph = np.linalg.cond(A_ph)
    print(f"  {N:4d}  {h:.6f}       {np.abs(phi-phi_exact).max():.3e}          "
          f"{phi.max():.6f}     {it:4d}      {np.sqrt(kappa_ph):7.2f}")
    assert np.abs(phi - phi_exact).max() < 1e-12
print(f"analytic peak rho0 d^2 / (8 eps0) = {rho0*d_slab**2/(8*eps0):.6f}")

   N       h        max|phi_num - phi_exact|   phi_max      CG iters   sqrt(kappa)


    15  0.062500       0.000e+00          0.125000        8        10.15
    31  0.031250       0.000e+00          0.125000       16        20.36
    63  0.015625       0.000e+00          0.125000       32        40.74
   127  0.007812       0.000e+00          0.125000       64        81.48
analytic peak rho0 d^2 / (8 eps0) = 0.125000


### Problem L2.8 — Implicit Euler for the heat equation is well conditioned (physics)

**Statement.** The heat equation $u_t = a\,u_{xx}$ on $[0,1]$ with $u(0,t)=u(1,t)=0$, discretized
in space by $A = \operatorname{tridiag}(-1,2,-1)/h^2$ and in time by implicit Euler, gives
$(I + r A h^2)u^{n+1} = u^{n}$ with $r = a\,\Delta t/h^{2}$. Show the step matrix is symmetric
positive definite with $\kappa_2 \le 1 + 4r$, and contrast with the explicit scheme.

**Intuition.** Adding the identity shifts the whole spectrum away from zero, and it is the near-zero
end of the spectrum that makes $A$ ill conditioned.

**Solution.**

*Step 1.* Write $B = I + rT$ with $T = \operatorname{tridiag}(-1,2,-1)$. From Problem L1.7 the
eigenvalues of $T$ are $\mu_k = 4\sin^2\bigl(k\pi/(2(N+1))\bigr) \in (0,4)$.

*Step 2.* Hence $B$ is symmetric with eigenvalues $1 + r\mu_k \gt 1 \gt 0$: symmetric positive
definite, so CG applies to every time step.

*Step 3.* The condition number is

$$
\kappa_2(B) = \frac{1 + r\mu_{\max}}{1 + r\mu_{\min}} \ \lt \ 1 + r\mu_{\max} \ \lt \ 1 + 4r .
$$

*Step 4.* If the time step is chosen parabolically, $\Delta t \propto h^2$, then $r$ is a
constant and $\kappa_2(B)$ is **bounded independently of the mesh**: the CG iteration count per
time step does not grow as the grid is refined.

*Step 5 — contrast.* Explicit Euler needs no solve but is stable only for
$r \le \tfrac12$, since its amplification factors are $1 - r\mu_k$ and $\mu_{\max} \to 4$.
Implicit Euler has amplification $1/(1+r\mu_k) \in (0,1)$ for every $r \gt 0$: unconditionally
stable.

$$
\boxed{\kappa_2(I + rT) = \frac{1+r\mu_{\max}}{1+r\mu_{\min}} \lt 1 + 4r, \qquad \text{explicit stability needs } r \le \tfrac12}
$$

**Key takeaway.** Implicit time stepping converts a stability restriction into a linear solve,
and the solve is easy precisely because the identity dominates: the physics of diffusion produces
a matrix CG likes.

In [32]:
N_h = 63
T_h = (np.diag(2.0 * np.ones(N_h)) + np.diag(-np.ones(N_h - 1), 1)
       + np.diag(-np.ones(N_h - 1), -1))
mu = np.linalg.eigvalsh(T_h)
print(f"N = {N_h}: mu_min = {mu[0]:.6e},  mu_max = {mu[-1]:.6f},  kappa_2(T) = {mu[-1]/mu[0]:.1f}")
print("    r     kappa_2(I + rT)   bound 1 + 4r   CG iters per step   explicit stable?")
u0 = rng.standard_normal(N_h)          # a rough profile excites every mode
for r in (0.25, 0.5, 1.0, 10.0):
    B = np.eye(N_h) + r * T_h
    _, it = cg_count(B, u0, tol=1e-12)
    print(f"  {r:5.2f}      {np.linalg.cond(B):10.4f}      {1+4*r:8.2f}          {it:4d}"
          f"             {'yes' if r <= 0.5 else 'no'}")
    assert np.linalg.cond(B) < 1 + 4 * r
    assert np.linalg.eigvalsh(B).min() > 1.0
print("mesh independence at fixed r = 1: kappa_2 and the CG count do not grow with N")
for N2 in (31, 63, 127):
    T2 = (np.diag(2.0 * np.ones(N2)) + np.diag(-np.ones(N2 - 1), 1)
          + np.diag(-np.ones(N2 - 1), -1))
    B2 = np.eye(N2) + 1.0 * T2
    _, it2 = cg_count(B2, rng.standard_normal(N2), tol=1e-12)
    print(f"    N = {N2:4d}:  kappa_2(I + T) = {np.linalg.cond(B2):.4f}   CG iters = {it2}")
# explicit Euler blows up beyond r = 1/2
for r in (0.5, 0.51):
    amp = np.abs(1 - r * mu).max()
    u = u0.copy()
    for _ in range(200):
        u = u - r * (T_h @ u)
    print(f"  explicit r = {r}: max|1 - r mu| = {amp:.6f}   ||u|| after 200 steps = "
          f"{np.linalg.norm(u):.3e}   (started at {np.linalg.norm(u0):.3e})")

N = 63: mu_min = 2.409088e-03,  mu_max = 3.997591,  kappa_2(T) = 1659.4
    r     kappa_2(I + rT)   bound 1 + 4r   CG iters per step   explicit stable?
   0.25          1.9982          2.00            16             yes
   0.50          2.9952          3.00            22             yes
   1.00          4.9856          5.00            29             no
  10.00         40.0120         41.00            63             no
mesh independence at fixed r = 1: kappa_2 and the CG count do not grow with N


    N =   31:  kappa_2(I + T) = 4.9428   CG iters = 28


    N =   63:  kappa_2(I + T) = 4.9856   CG iters = 29
    N =  127:  kappa_2(I + T) = 4.9964   CG iters = 29
  explicit r = 0.5: max|1 - r mu| = 0.998795   ||u|| after 200 steps = 8.778e-01   (started at 7.969e+00)
  explicit r = 0.51: max|1 - r mu| = 1.038771   ||u|| after 200 steps = 1.722e+03   (started at 7.969e+00)


### Problem L2.9 — K-FAC as a Kronecker preconditioner

**Statement.** For a linear layer $y = Wa$ with $W \in \mathbb{R}^{d_1 \times d_2}$, the Fisher
matrix is $F = \mathbb{E}[\operatorname{vec}(\nabla_W \mathcal{L})\operatorname{vec}(\nabla_W \mathcal{L})^{\top}]$.
Show that the K-FAC approximation $F \approx A \otimes B$ reduces the cost of applying $F^{-1}$
from $O(d_1^3 d_2^3)$ to $O(d_1^3 + d_2^3)$.

**Intuition.** A layer gradient is an outer product of an activation and a backpropagated
signal; pretending the two factors are independent turns the Fisher into a Kronecker product,
which inverts factor by factor.

**Solution.**

*Step 1.* With $s = \nabla_y \mathcal{L}$ and activation $a$, the layer gradient is
$\nabla_W \mathcal{L} = s a^{\top}$, so $\operatorname{vec}(\nabla_W\mathcal{L}) = a \otimes s$.

*Step 2.* Hence
$F = \mathbb{E}[(a \otimes s)(a \otimes s)^{\top}] = \mathbb{E}[(aa^{\top}) \otimes (ss^{\top})]$.

*Step 3.* K-FAC replaces the expectation of the product by the product of the expectations,

$$
F \approx A \otimes B, \qquad A = \mathbb{E}[aa^{\top}] \in \mathbb{R}^{d_2 \times d_2}, \quad B = \mathbb{E}[ss^{\top}] \in \mathbb{R}^{d_1 \times d_1}.
$$

*Step 4.* $(A \otimes B)^{-1} = A^{-1} \otimes B^{-1}$, and
$(A^{-1}\otimes B^{-1})\operatorname{vec}(G) = \operatorname{vec}(B^{-1}GA^{-1})$.

*Step 5.* Inverting $F$ directly costs $O((d_1d_2)^3)$; inverting $A$ and $B$ costs
$O(d_2^3) + O(d_1^3)$.

$$
\boxed{F^{-1}\operatorname{vec}(G) \approx \operatorname{vec}(B^{-1}GA^{-1}), \qquad O(d_1^3 + d_2^3)}
$$

**Key takeaway.** This is Definition 3.10 with a structured $M$: the requirement is only that
$M^{-1}v$ be cheap, and Kronecker structure is one of the cheapest ways to arrange that.

In [33]:
d1, d2 = 6, 5
A_k = rng.standard_normal((d2, d2))
A_k = A_k @ A_k.T + d2 * np.eye(d2)
B_k = rng.standard_normal((d1, d1))
B_k = B_k @ B_k.T + d1 * np.eye(d1)
F = np.kron(A_k, B_k)
G = rng.standard_normal((d1, d2))
direct = np.linalg.solve(F, G.reshape(-1, order="F"))
kfac = (np.linalg.solve(B_k, G) @ np.linalg.inv(A_k)).reshape(-1, order="F")
print("||F^-1 vec(G) - vec(B^-1 G A^-1)|| :", np.linalg.norm(direct - kfac))
print(f"flops: direct (d1 d2)^3 = {(d1*d2)**3},   K-FAC d1^3 + d2^3 = {d1**3 + d2**3}")
for dd in (64, 256, 1024):
    print(f"  d1 = d2 = {dd:5d}:  ratio = {(dd*dd)**3/(2*dd**3):.3e}")
assert np.linalg.norm(direct - kfac) < 1e-9

||F^-1 vec(G) - vec(B^-1 G A^-1)|| : 2.5877705110315343e-17
flops: direct (d1 d2)^3 = 27000,   K-FAC d1^3 + d2^3 = 341
  d1 = d2 =    64:  ratio = 1.311e+05
  d1 = d2 =   256:  ratio = 8.389e+06
  d1 = d2 =  1024:  ratio = 5.369e+08


### Problem L2.10 — Randomized Nystrom preconditioning for kernel ridge regression

**Statement.** For $(K + \mu I)x = y$ with $K$ symmetric positive semidefinite, build the
Nystrom preconditioner $P = \tilde{K} + \mu I$ from a sketch $S \in \mathbb{R}^{n \times s}$, and
bound $\kappa_2\bigl(P^{-1/2}(K+\mu I)P^{-1/2}\bigr)$.

**Intuition.** Kernel spectra decay fast, so a rank-$s$ sketch captures nearly all of $K$; what
it misses is smaller than $\lambda_{s+1}$, and the ridge $\mu$ already dominates that.

**Solution.**

*Step 1.* The Nystrom approximation is
$\tilde{K} = (KS)(S^{\top}KS)^{+}(S^{\top}K)$, of rank at most $s$, with
$0 \preceq \tilde K \preceq K$.

*Step 2.* Set $P = \tilde{K} + \mu I$. By the Woodbury identity $P^{-1}v$ costs one $n \times s$
product and one $s \times s$ solve, so $O(ns + s^3)$.

*Step 3.* Write $E = K - \tilde{K} \succeq 0$. Then
$P^{-1/2}(K + \mu I)P^{-1/2} = I + P^{-1/2}EP^{-1/2}$, whose eigenvalues lie in
$[1, 1 + \lambda_{\max}(E)/\mu]$ because $P \succeq \mu I$.

*Step 4.* Hence

$$
\kappa_2\bigl(P^{-1/2}(K+\mu I)P^{-1/2}\bigr) \ \le\ 1 + \frac{\lambda_{\max}(K - \tilde K)}{\mu},
$$

and for a sketch capturing the top $s$ eigenspaces $\lambda_{\max}(K-\tilde K) \approx \lambda_{s+1}(K)$.

$$
\boxed{\kappa_2\bigl(P^{-1/2}(K+\mu I)P^{-1/2}\bigr) \le 1 + \frac{\lambda_{\max}(K - \tilde{K})}{\mu}}
$$

**Key takeaway.** The bound is independent of $n$ once the spectrum decays, so preconditioned CG
solves kernel ridge regression in a number of iterations set by the *tail* of the spectrum, not
by its extremes.

In [34]:
n_ker = 300
pts = np.sort(rng.uniform(0, 1, n_ker))
K = np.exp(-((pts[:, None] - pts[None, :]) ** 2) / (2 * 0.05 ** 2))
mu_ridge = 1e-3
s_sk = 40
S = rng.standard_normal((n_ker, s_sk))
KS = K @ S
Ktil = KS @ np.linalg.pinv(S.T @ K @ S) @ KS.T
Ktil = (Ktil + Ktil.T) / 2
E = K - Ktil
P = Ktil + mu_ridge * np.eye(n_ker)
w, V = np.linalg.eigh(P)
P_isqrt = V @ np.diag(w ** -0.5) @ V.T
A_prec = P_isqrt @ (K + mu_ridge * np.eye(n_ker)) @ P_isqrt
lam_K = np.linalg.eigvalsh(K)[::-1]
print(f"kappa_2(K + mu I)                 : {np.linalg.cond(K + mu_ridge*np.eye(n_ker)):.3e}")
print(f"kappa_2 after Nystrom (s = {s_sk})    : {np.linalg.cond(A_prec):.3f}")
print(f"bound 1 + lambda_max(K - Ktil)/mu : {1 + np.linalg.eigvalsh(E).max()/mu_ridge:.3f}")
print(f"lambda_{s_sk+1}(K) = {lam_K[s_sk]:.3e}")
y_ker = rng.standard_normal(n_ker)
_, it_plain = cg_count(K + mu_ridge * np.eye(n_ker), y_ker, tol=1e-8)
_, it_prec = cg_count(A_prec, P_isqrt @ y_ker, tol=1e-8)
print(f"CG iterations: unpreconditioned {it_plain}, Nystrom-preconditioned {it_prec}")
assert np.linalg.cond(A_prec) <= 1 + np.linalg.eigvalsh(E).max() / mu_ridge + 1e-6
assert it_prec < it_plain

kappa_2(K + mu I)                 : 3.886e+04
kappa_2 after Nystrom (s = 40)    : 1.020


bound 1 + lambda_max(K - Ktil)/mu : 1.084
lambda_41(K) = 1.148e-06
CG iterations: unpreconditioned 118, Nystrom-preconditioned 3


### Problem L2.11 — PageRank as a linear system

**Statement.** With $P$ column-stochastic and damping $\alpha \in (0,1)$, PageRank solves
$(I - \alpha P)x = \tfrac{1-\alpha}{n}\mathbf{1}$. Show that the Jacobi iteration converges with
rate at most $\alpha$, independently of $n$.

**Intuition.** Damping shrinks every transition probability by $\alpha$, so the whole spectrum of
the iteration matrix is inside a disc of radius $\alpha$.

**Solution.**

*Step 1.* The diagonal of $I - \alpha P$ is $1 - \alpha p_{ii}$; for the plain power form take
$M = I$, giving the iteration

$$
x_{k+1} = \alpha P x_k + \frac{1-\alpha}{n}\mathbf{1}, \qquad G = \alpha P .
$$

*Step 2.* $P$ is column-stochastic, so its columns are non-negative and sum to $1$, hence
$\lVert P \rVert_1 = \max_j \sum_i p_{ij} = 1$.

*Step 3.* Therefore $\lVert G \rVert_1 = \alpha$ and $\rho(G) \le \lVert G \rVert_1 = \alpha$.

*Step 4.* By Theorem 4.2 the iteration converges, with
$\lVert e_k \rVert_1 \le \alpha^{k}\lVert e_0 \rVert_1$. The bound does not mention $n$.

$$
\boxed{\rho(\alpha P) \le \alpha, \qquad \lVert e_k \rVert_1 \le \alpha^{k}\lVert e_0 \rVert_1}
$$

**Key takeaway.** At $\alpha = 0.85$, reaching $10^{-8}$ takes $\lceil 8\ln 10 / \ln(1/0.85)\rceil$
steps whether the web graph has $10^3$ or $10^{10}$ nodes. Damping is a preconditioner in
disguise: it moves the spectrum away from $1$.

In [35]:
n_pr = 500
Aadj = (rng.random((n_pr, n_pr)) < 0.02).astype(float)
Aadj[Aadj.sum(axis=0) == 0, 0] = 1.0
P_pr = Aadj / Aadj.sum(axis=0)
alpha = 0.85
x_pr = np.ones(n_pr) / n_pr
x_true = np.linalg.solve(np.eye(n_pr) - alpha * P_pr, (1 - alpha) / n_pr * np.ones(n_pr))
errs = [np.abs(x_pr - x_true).sum()]
for _ in range(60):
    x_pr = alpha * P_pr @ x_pr + (1 - alpha) / n_pr * np.ones(n_pr)
    errs.append(np.abs(x_pr - x_true).sum())
errs = np.array(errs)
print("column sums of P (min, max):", P_pr.sum(axis=0).min(), P_pr.sum(axis=0).max())
print("||alpha P||_1 :", alpha * np.abs(P_pr).sum(axis=0).max())
print("rho(alpha P)  :", max(abs(np.linalg.eigvals(alpha * P_pr))))
print("observed contraction over 60 steps:", (errs[60] / errs[0]) ** (1 / 60))
print("steps predicted for 1e-8:", int(np.ceil(8 * np.log(10) / np.log(1 / alpha))))
print(f"error after 60 steps: {errs[-1]:.3e}   x sums to {x_pr.sum():.6f}")
assert max(abs(np.linalg.eigvals(alpha * P_pr))) <= alpha + 1e-12
assert (errs[60] / errs[0]) ** (1 / 60) <= alpha + 1e-9

column sums of P (min, max): 0.9999999999999996 1.0000000000000004
||alpha P||_1 : 0.8500000000000003


rho(alpha P)  :

 0.8500000000000003
observed contraction over 60 steps: 0.5655541415367915
steps predicted for 1e-8: 114
error after 60 steps: 3.442e-16   x sums to 1.000000


### Problem L2.12 — Krylov solvers for a Schrodinger-type operator (physics)

**Statement.** Let $\mathcal{A}u = -\Delta u + V(x)u$ on $L^2(\Omega)$ satisfy the coercivity
bound $\langle \mathcal{A}v, v \rangle \ge \gamma \lVert v \rVert^2$ with $\gamma \gt 0$ and the
continuity bound $\lVert \mathcal{A}v \rVert \le C\lVert v \rVert$ on the relevant subspace.
Derive the residual bound for the Krylov minimization
$u_k = \operatorname{argmin}\{\lVert f - \mathcal{A}v \rVert : v \in \mathcal{K}_k(\mathcal{A}, f)\}$.

**Intuition.** Coercivity says the operator never turns a vector by more than a right angle, so a
single optimally scaled step always removes a fixed fraction of the residual.

**Solution.**

*Step 1 — one step.* Minimize over the single direction $f$:

$$
\lVert r_1 \rVert^2 = \min_{\tau} \lVert f - \tau \mathcal{A}f \rVert^2
= \lVert f \rVert^2 - \frac{\langle \mathcal{A}f, f\rangle^{2}}{\lVert \mathcal{A}f \rVert^{2}},
$$

the minimum being at $\tau^{\star} = \langle \mathcal{A}f, f\rangle / \lVert \mathcal{A}f \rVert^2$.

*Step 2 — use the two bounds.* Writing $v_0 = f/\lVert f \rVert$, coercivity gives
$\langle \mathcal{A}v_0, v_0 \rangle \ge \gamma$ and continuity gives
$\lVert \mathcal{A}v_0 \rVert \le C$, so

$$
\frac{\langle \mathcal{A}v_0, v_0\rangle^{2}}{\lVert \mathcal{A}v_0 \rVert^{2}} \ \ge\ \frac{\gamma^{2}}{C^{2}} .
$$

*Step 3.* Hence $\lVert r_1 \rVert^2 \le (1 - \gamma^2/C^2)\lVert f \rVert^2$.

*Step 4 — iterate.* The $k$-step minimization ranges over a superset of what $k$ successive
one-step minimizations reach, so its residual is no larger:

$$
\frac{\lVert r_k \rVert}{\lVert f \rVert} \le \left(1 - \frac{\gamma^{2}}{C^{2}}\right)^{k/2}.
$$

$$
\boxed{\frac{\lVert f - \mathcal{A}u_k \rVert}{\lVert f \rVert} \le \left(1 - \frac{\gamma^{2}}{C^{2}}\right)^{k/2}}
$$

**Key takeaway.** This is Elman's bound (Problem L3.4) transcribed to an operator: the ratio
$\gamma/C$ is the continuous analogue of a condition number, and it is what a physics-informed
solver must control.

The check below makes one point the derivation hides. The differential operator is *unbounded*,
so the raw discretization has $C \sim h^{-2}$ and the bound, while true, is vacuous. Posing the
problem in the energy norm — equivalently, preconditioning symmetrically by the Laplacian — makes
$\gamma$ and $C$ order one and the bound informative.

In [36]:
N_op = 120
h_op = 1.0 / (N_op + 1)
xs_op = np.arange(1, N_op + 1) * h_op
V_pot = 30.0 * np.exp(-((xs_op - 0.5) ** 2) / (2 * 0.1 ** 2))
Lap = (np.diag(2.0 * np.ones(N_op)) + np.diag(-np.ones(N_op - 1), 1)
       + np.diag(-np.ones(N_op - 1), -1)) / h_op ** 2
Aop_raw = Lap + np.diag(V_pot)
# The continuous operator is unbounded, so C = infinity and the bound is vacuous unless the
# problem is posed in the energy norm. Preconditioning symmetrically by the Laplacian is
# exactly that change of norm, and it makes gamma and C order one.
wL, VL = np.linalg.eigh(Lap)
L_isqrt = VL @ np.diag(wL ** -0.5) @ VL.T
Aop = L_isqrt @ Aop_raw @ L_isqrt
Aop = (Aop + Aop.T) / 2
f_op = L_isqrt @ np.sin(np.pi * xs_op)
sym = (Aop + Aop.T) / 2
gamma_op = np.linalg.eigvalsh(sym).min()
C_op = np.linalg.norm(Aop, 2)
print(f"unpreconditioned: gamma = {np.linalg.eigvalsh((Aop_raw+Aop_raw.T)/2).min():.3f}, "
      f"C = {np.linalg.norm(Aop_raw, 2):.1f}  ->  factor "
      f"{1 - np.linalg.eigvalsh((Aop_raw+Aop_raw.T)/2).min()**2/np.linalg.norm(Aop_raw,2)**2:.8f}")


def gmres_res(A, b, steps):
    n_ = len(b)
    Qg = np.zeros((n_, steps + 1))
    Hg = np.zeros((steps + 1, steps))
    beta = np.linalg.norm(b)
    Qg[:, 0] = b / beta
    out = [beta]
    for k in range(steps):
        w = A @ Qg[:, k]
        for i in range(k + 1):
            Hg[i, k] = Qg[:, i] @ w
            w = w - Hg[i, k] * Qg[:, i]
        Hg[k + 1, k] = np.linalg.norm(w)
        if Hg[k + 1, k] > 1e-14:
            Qg[:, k + 1] = w / Hg[k + 1, k]
        rr = np.zeros(k + 2)
        rr[0] = beta
        yy, *_ = np.linalg.lstsq(Hg[:k + 2, :k + 1], rr, rcond=None)
        out.append(np.linalg.norm(rr - Hg[:k + 2, :k + 1] @ yy))
    return np.array(out)


res_op = gmres_res(Aop, f_op, 40)
factor = 1 - gamma_op ** 2 / C_op ** 2
print(f"coercivity gamma = {gamma_op:.4f},   continuity C = ||A||_2 = {C_op:.4f}")
print(f"one-step factor (1 - gamma^2/C^2) = {factor:.6f}")
print("  k    measured ||r_k||/||f||    bound (1 - g^2/C^2)^(k/2)")
for k in (5, 10, 20, 40):
    print(f" {k:3d}      {res_op[k]/res_op[0]:.6e}          {factor**(k/2):.6e}")
    assert res_op[k] / res_op[0] <= factor ** (k / 2) + 1e-12

unpreconditioned: gamma = 22.274, C = 58569.0  ->  factor 0.99999986
coercivity gamma = 1.0000,   continuity C = ||A||_2 = 2.4721
one-step factor (1 - gamma^2/C^2) = 0.836373
  k    measured ||r_k||/||f||    bound (1 - g^2/C^2)^(k/2)
   5      3.611732e-10          6.397339e-01
  10      5.321339e-16          4.092595e-01
  20      7.147507e-16          1.674933e-01
  40      7.440804e-16          2.805401e-02


## L3 — Challenge Proofs

### Problem L3.1 — The relative distance to singularity is $1/\kappa_2(A)$

**Statement.** For non-singular $A \in \mathbb{R}^{n \times n}$ prove

$$
\min \left\lbrace \frac{\lVert \Delta A \rVert_{\mathrm{op}}}{\lVert A \rVert_{\mathrm{op}}} : A + \Delta A \text{ singular} \right\rbrace = \frac{1}{\kappa_2(A)} .
$$

**Intuition.** The smallest singular value measures how much room $A$ has before it collapses;
scaling it against the largest gives a relative measure.

**Solution.**

*Step 1 — a perturbation that achieves $\sigma_n$.* Let $A = U\Sigma V^{\top}$ with
$\sigma_1 \ge \dots \ge \sigma_n \gt 0$. Take $\Delta A = -\sigma_n u_n v_n^{\top}$. Then
$(A + \Delta A)v_n = \sigma_n u_n - \sigma_n u_n = 0$, so $A + \Delta A$ is singular, and
$\lVert \Delta A \rVert_{\mathrm{op}} = \sigma_n$.

*Step 2 — no perturbation does better.* Suppose $A + \Delta A$ is singular. Then
$(A+\Delta A)x = 0$ for some unit $x$, so $Ax = -\Delta A x$ and

$$
\sigma_n = \min_{\lVert z \rVert_2 = 1}\lVert Az \rVert_2 \le \lVert Ax \rVert_2 = \lVert \Delta A x \rVert_2 \le \lVert \Delta A \rVert_{\mathrm{op}} .
$$

*Step 3 — combine.* The minimum is exactly $\sigma_n$.

*Step 4 — normalize.* Dividing by $\lVert A \rVert_{\mathrm{op}} = \sigma_1$,

$$
\frac{\sigma_n}{\sigma_1} = \frac{1}{\sigma_1/\sigma_n} = \frac{1}{\kappa_2(A)} .
$$

$$
\boxed{\min_{A+\Delta A \text{ singular}} \frac{\lVert \Delta A \rVert_{\mathrm{op}}}{\lVert A \rVert_{\mathrm{op}}} = \frac{1}{\kappa_2(A)}}
$$

**Key takeaway.** This makes the hypothesis of Theorem 4.1 sharp: the condition
$\lVert A^{-1} \rVert \lVert \Delta A \rVert \lt 1$ says precisely that $\Delta A$ is smaller than
the distance to the nearest singular matrix.

In [37]:
A_d = rng.standard_normal((6, 6))
Ud, Sd, Vtd = np.linalg.svd(A_d)
dA = -Sd[-1] * np.outer(Ud[:, -1], Vtd[-1])
print(f"sigma_min = {Sd[-1]:.6f},  ||dA||_2 = {np.linalg.norm(dA, 2):.6f}")
print(f"smallest singular value of A + dA : {np.linalg.svd(A_d + dA, compute_uv=False)[-1]:.3e}")
print(f"relative distance = {np.linalg.norm(dA,2)/np.linalg.norm(A_d,2):.8f}"
      f"   1/kappa_2(A) = {1/np.linalg.cond(A_d):.8f}")
for _ in range(200):
    E = rng.standard_normal((6, 6))
    E = E / np.linalg.norm(E, 2) * (0.999 * Sd[-1])
    assert np.linalg.svd(A_d + E, compute_uv=False)[-1] > 0
print("200 random perturbations of norm 0.999 sigma_min all left A + E non-singular")
assert abs(np.linalg.norm(dA, 2) / np.linalg.norm(A_d, 2) - 1 / np.linalg.cond(A_d)) < 1e-12

sigma_min = 0.514469,  ||dA||_2 = 0.514469
smallest singular value of A + dA : 6.723e-16
relative distance = 0.11015864   1/kappa_2(A) = 0.11015864
200 random perturbations of norm 0.999 sigma_min all left A + E non-singular


### Problem L3.2 — Minimax optimality of the scaled Chebyshev polynomial

**Statement.** Let $0 \lt a \le b$ and $\mathcal{P}_k^1 = \{p \in \mathcal{P}_k : p(0)=1\}$. Prove
that

$$
p_k^{\star}(\lambda) = \frac{T_k\bigl(\ell(\lambda)\bigr)}{T_k(w)}, \qquad
\ell(\lambda) = \frac{b+a-2\lambda}{b-a}, \quad w = \frac{b+a}{b-a},
$$

is the unique minimizer of $\max_{[a,b]}\lvert p \rvert$ over $\mathcal{P}_k^1$, and deduce the CG
bound $2\bigl((\sqrt\kappa-1)/(\sqrt\kappa+1)\bigr)^k$ with $\kappa = b/a$.

**Intuition.** $T_k$ is as flat as a degree-$k$ polynomial can be on $[-1,1]$ and grows faster
than any competitor outside it, so normalizing it at the outside point $w$ makes it as small as
possible inside.

**Solution.**

*Step 1 — the candidate is admissible.* $\ell$ maps $[a,b]$ onto $[-1,1]$ and $0$ to $w \gt 1$;
$T_k$ is increasing on $[1,\infty)$ with $T_k(1)=1$, so $T_k(w) \gt 1$ and $p_k^{\star}(0)=1$.
Since $\lvert T_k \rvert \le 1$ on $[-1,1]$, $M := \max_{[a,b]}\lvert p_k^{\star} \rvert = 1/T_k(w)$.

*Step 2 — equioscillation.* $T_k(\cos\theta) = \cos k\theta$, so $T_k$ attains $(-1)^j$ at
$y_j = \cos(j\pi/k)$, $j = 0,\dots,k$. Let $\lambda_j = \ell^{-1}(y_j) \in [a,b]$; these are $k+1$
distinct points with $p_k^{\star}(\lambda_j) = (-1)^{j}M$.

*Step 3 — suppose a competitor beats it.* Let $q \in \mathcal{P}_k^1$ with
$\max_{[a,b]}\lvert q \rvert \lt M$, and set $d = p_k^{\star} - q$, of degree at most $k$.

*Step 4 — count sign changes.* At $\lambda_j$, $d(\lambda_j) = (-1)^{j}M - q(\lambda_j)$ has the
sign of $(-1)^{j}$, because $\lvert q(\lambda_j)\rvert \lt M$. So $d$ changes sign in each of the
$k$ open intervals between consecutive $\lambda_j$, giving $k$ distinct roots.

*Step 5 — one more root.* $d(0) = 1 - 1 = 0$, and $0 \notin [a,b]$ since $a \gt 0$, so this root
is distinct from the previous $k$.

*Step 6 — contradiction.* A polynomial of degree at most $k$ with $k+1$ distinct roots vanishes
identically, so $q = p_k^{\star}$, contradicting $\max\lvert q\rvert \lt M$. The minimum value
is therefore $1/T_k(w)$; that the minimizer is unique is the equality case of the same count and
is carried out in Cheney, *Introduction to Approximation Theory*, 2nd ed., chapter 3, Theorem 1.

*Step 7 — evaluate.* With $\kappa = b/a$, $w = (\kappa+1)/(\kappa-1)$ and
$w + \sqrt{w^2-1} = (\sqrt\kappa+1)/(\sqrt\kappa-1)$, so from
$T_k(y) = \tfrac12\bigl[(y+\sqrt{y^2-1})^k + (y-\sqrt{y^2-1})^k\bigr] \ge \tfrac12 (y+\sqrt{y^2-1})^k$,

$$
M = \frac{1}{T_k(w)} \le 2\left(\frac{\sqrt\kappa - 1}{\sqrt\kappa+1}\right)^{k} .
$$

$$
\boxed{\min_{p \in \mathcal{P}_k^{1}}\max_{[a,b]}\lvert p \rvert = \frac{1}{T_k(w)} \le 2\left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^{k}}
$$

**Key takeaway.** This is Lemma 5.6.1, and it is the only place where the square root in
Theorem 4.7 comes from. The equioscillation count is the proof; asserting that "Chebyshev
polynomials grow fastest outside $[-1,1]$" is not.

In [38]:
from numpy.polynomial import chebyshev as npcheb

a_c, b_c, k_c = 2.0, 30.0, 5
w_c = (b_c + a_c) / (b_c - a_c)
Tw = npcheb.chebval(w_c, [0] * k_c + [1])
lam_c = np.linspace(a_c, b_c, 4000)
p_star = npcheb.chebval((b_c + a_c - 2 * lam_c) / (b_c - a_c), [0] * k_c + [1]) / Tw
M_star = np.abs(p_star).max()
print(f"1/T_k(w) = {1/Tw:.8f}   max|p*| = {M_star:.8f}")
# perturb the optimum by a random polynomial vanishing at 0, so p(0) = 1 is preserved
best = np.inf
for _ in range(4000):
    coef = rng.standard_normal(k_c) * 0.02
    d_poly = sum(coef[j] * (lam_c / b_c) ** (j + 1) for j in range(k_c))
    best = min(best, np.abs(p_star + d_poly).max())
print(f"best of 4000 perturbations of p* that keep p(0) = 1 : {best:.8f}")
print(f"every perturbation is worse : {best >= M_star}")
kap_c = b_c / a_c
print(f"Chebyshev bound 2((sqrt k -1)/(sqrt k +1))^k = "
      f"{2*((np.sqrt(kap_c)-1)/(np.sqrt(kap_c)+1))**k_c:.8f}")
assert abs(M_star - 1 / Tw) < 1e-10
assert best >= M_star - 1e-12
assert M_star <= 2 * ((np.sqrt(kap_c) - 1) / (np.sqrt(kap_c) + 1)) ** k_c

1/T_k(w) = 0.14174988   max|p*| = 0.14174988


best of 4000 perturbations of p* that keep p(0) = 1 : 0.14190488
every perturbation is worse : True
Chebyshev bound 2((sqrt k -1)/(sqrt k +1))^k = 0.14246917


### Problem L3.3 — Stein-Rosenberg: Gauss-Seidel beats Jacobi on an $M$-matrix

**Statement.** Let $A = D - L - U$ with $D$ positive diagonal and $L, U \ge 0$ entrywise, so that
$G_{\mathrm{J}} = D^{-1}(L+U) \ge 0$. Prove that $\rho(G_{\mathrm{J}}) \lt 1$ implies
$\rho(G_{\mathrm{GS}}) \le \rho(G_{\mathrm{J}})$.

**Intuition.** With non-negative off-diagonals, using the freshly updated components can only
help, because every contribution has the same sign.

**Solution.**

*Step 1 — Perron eigenvector.* $G_{\mathrm{J}} \ge 0$, so by Perron-Frobenius there is
$v \ge 0$, $v \neq 0$, with $G_{\mathrm{J}}v = \mu v$ and $\mu = \rho(G_{\mathrm{J}}) \lt 1$.
Equivalently $(L+U)v = \mu D v$.

*Step 2 — compare the two operators on $v$.* Compute

$$
G_{\mathrm{GS}}v - \mu v = (D-L)^{-1}\bigl[ Uv - \mu(D-L)v \bigr] .
$$

*Step 3 — simplify the bracket.* Substituting $Uv = \mu Dv - Lv$ from Step 1,

$$
Uv - \mu(D-L)v = (\mu Dv - Lv) - \mu Dv + \mu Lv = -(1-\mu)Lv \ \le\ 0 ,
$$

because $\mu \lt 1$, $L \ge 0$ and $v \ge 0$.

*Step 4 — $(D-L)^{-1}$ preserves sign.* Since $D^{-1}L \ge 0$ is strictly lower triangular it is
nilpotent, so

$$
(D-L)^{-1} = \bigl(I - D^{-1}L\bigr)^{-1}D^{-1} = \left( \sum_{j=0}^{n-1}(D^{-1}L)^{j} \right) D^{-1} \ \ge\ 0 .
$$

*Step 5 — conclude.* Therefore $G_{\mathrm{GS}}v \le \mu v$ with $v \ge 0$, $v \neq 0$. The
Collatz-Wielandt characterization of the Perron root of a non-negative matrix gives

$$
\rho(G_{\mathrm{GS}}) \le \mu = \rho(G_{\mathrm{J}}) .
$$

*Step 6 — strictness.* Strict inequality $\rho(G_{\mathrm{GS}}) \lt \rho(G_{\mathrm{J}})$ requires
$G_{\mathrm{J}}$ to be irreducible and is the full Stein-Rosenberg theorem; see Varga,
*Matrix Iterative Analysis*, 2nd ed., Theorem 3.15.

$$
\boxed{\rho(G_{\mathrm{GS}}) \le \rho(G_{\mathrm{J}}) \lt 1, \text{ with strict inequality when } G_{\mathrm{J}} \text{ is irreducible}}
$$

**Key takeaway.** The sign structure is doing the work. Drop it — as in the positive definite
matrix of Section 7.4, whose off-diagonals are positive so that $L, U \le 0$ — and Jacobi can
diverge while Gauss-Seidel converges, or the comparison can fail in either direction.

In [39]:
for trial in range(4):
    n_sr = 5
    off = rng.random((n_sr, n_sr)) * 0.3
    np.fill_diagonal(off, 0.0)
    A_sr = np.diag(off.sum(axis=1) + rng.uniform(0.5, 1.0, n_sr)) - off
    D_sr = np.diag(np.diag(A_sr))
    L_sr = -np.tril(A_sr, -1)
    U_sr = -np.triu(A_sr, 1)
    rj = max(abs(np.linalg.eigvals(np.linalg.solve(D_sr, L_sr + U_sr))))
    rg = max(abs(np.linalg.eigvals(np.linalg.solve(D_sr - L_sr, U_sr))))
    print(f"  trial {trial}: L, U >= 0 -> rho(G_J) = {rj:.6f}   rho(G_GS) = {rg:.6f}   "
          f"GS faster: {rg < rj}")
    assert L_sr.min() >= 0 and U_sr.min() >= 0
    assert rg <= rj + 1e-12

  trial 0: L, U >= 0 -> rho(G_J) = 0.487159   rho(G_GS) = 0.261503   GS faster: True
  trial 1: L, U >= 0 -> rho(G_J) = 0.395353   rho(G_GS) = 0.148570   GS faster: True
  trial 2: L, U >= 0 -> rho(G_J) = 0.406650   rho(G_GS) = 0.186339   GS faster: True
  trial 3: L, U >= 0 -> rho(G_J) = 0.442385   rho(G_GS) = 0.236565   GS faster: True


### Problem L3.4 — Elman's field-of-values bound for GMRES

**Statement.** Let $W(A) = \{x^{\ast}Ax : \lVert x \rVert_2 = 1\}$ and suppose
$d = \operatorname{dist}(0, W(A)) \gt 0$. Prove

$$
\frac{\lVert r_k \rVert_2}{\lVert r_0 \rVert_2} \le \left( 1 - \frac{d^{2}}{\lVert A \rVert_{\mathrm{op}}^{2}} \right)^{k/2}.
$$

**Intuition.** If the origin is outside the numerical range, then $A$ never rotates a vector by a
right angle, so one steepest-descent-style step always removes a fixed fraction of the residual.

**Solution.**

*Step 1 — the one-step minimum.* Restricting the GMRES minimization to
$x_1 = x_0 + \tau r_0$,

$$
\lVert r_1 \rVert_2^2 \ \le\ \min_{\tau}\lVert r_0 - \tau A r_0 \rVert_2^2
= \lVert r_0 \rVert_2^2 - \frac{\lvert r_0^{\ast}Ar_0 \rvert^{2}}{\lVert Ar_0 \rVert_2^{2}} ,
$$

the minimizer being $\tau^{\star} = r_0^{\ast}Ar_0 / \lVert Ar_0 \rVert_2^2$.

*Step 2 — use the two hypotheses.* With $v_0 = r_0/\lVert r_0 \rVert_2$,
$\lvert v_0^{\ast}Av_0 \rvert \ge d$ by definition of $d$, and
$\lVert Av_0 \rVert_2 \le \lVert A \rVert_{\mathrm{op}}$.

*Step 3 — one-step contraction.*

$$
\lVert r_1 \rVert_2^2 \le \left(1 - \frac{d^{2}}{\lVert A \rVert_{\mathrm{op}}^{2}}\right)\lVert r_0 \rVert_2^{2} .
$$

*Step 4 — chain the steps legitimately.* Let $\hat{x}_j$ be the iterates of restarted GMRES(1).
By induction $\hat{x}_j \in x_0 + \mathcal{K}_j(A, r_0)$: true at $j=1$, and
$\hat{x}_{j+1} = \hat{x}_j + \tau_j \hat{r}_j$ with $\hat{r}_j \in \mathcal{K}_{j+1}$ gives
$\hat{x}_{j+1} \in x_0 + \mathcal{K}_{j+1}$.

*Step 5 — conclude.* Full GMRES minimizes over $x_0 + \mathcal{K}_k$, which contains
$\hat{x}_k$, so $\lVert r_k \rVert_2 \le \lVert \hat{r}_k \rVert_2 \le (1 - d^2/\lVert A \rVert_{\mathrm{op}}^2)^{k/2}\lVert r_0 \rVert_2$.

$$
\boxed{\frac{\lVert r_k \rVert_2}{\lVert r_0 \rVert_2} \le \left(1 - \frac{d^{2}}{\lVert A \rVert_{\mathrm{op}}^{2}}\right)^{k/2}}
$$

**Key takeaway.** Unlike Theorem 4.8 part 4 this bound never mentions $\kappa_2(V)$, so it
survives non-normality — at the price of being useless when $0$ is inside or near $W(A)$, which is
exactly the cyclic-shift situation of Problem L3.8.

In [40]:
m_fv = 40
A_fv = rng.standard_normal((m_fv, m_fv)) / np.sqrt(m_fv) + 3.0 * np.eye(m_fv)
b_fv = rng.standard_normal(m_fv)
sym_part = (A_fv + A_fv.T) / 2
d_fv = np.linalg.eigvalsh(sym_part).min()
nA = np.linalg.norm(A_fv, 2)
res_fv = gmres_res(A_fv, b_fv, 20)
fac = 1 - d_fv ** 2 / nA ** 2
print(f"dist(0, W(A)) >= lambda_min of the symmetric part = {d_fv:.6f}")
print(f"||A||_2 = {nA:.6f},   one-step factor = {fac:.6f}")
print("  k    measured      bound")
for k in (1, 5, 10, 20):
    print(f" {k:3d}   {res_fv[k]/res_fv[0]:.6e}   {fac**(k/2):.6e}")
    assert res_fv[k] / res_fv[0] <= fac ** (k / 2) + 1e-12

dist(0, W(A)) >= lambda_min of the symmetric part = 1.704172
||A||_2 = 4.409806,   one-step factor = 0.850656
  k    measured      bound
   1   2.975139e-01   9.223101e-01
   5   2.840555e-03   6.673978e-01
  10   4.313668e-06   4.454199e-01
  20   4.780571e-12   1.983989e-01


### Problem L3.5 — Superlinear CG under a low-rank perturbation

**Statement.** Let $A = I + K$ be symmetric positive definite with $K$ symmetric of rank
$r \lt n$. Prove that CG reaches the exact solution in at most $r+1$ iterations.

**Intuition.** A rank-$r$ perturbation moves at most $r$ eigenvalues off $1$, so the spectrum has
at most $r+1$ distinct points and one degree-$(r+1)$ polynomial annihilates it.

**Solution.**

*Step 1 — count the distinct eigenvalues.* $K$ symmetric of rank $r$ has exactly $n-r$ zero
eigenvalues, so $A = I+K$ has the eigenvalue $1$ with multiplicity at least $n-r$ and at most $r$
other eigenvalues $1+\mu_1, \dots, 1+\mu_r$. Hence $A$ has $d \le r+1$ distinct eigenvalues.

*Step 2 — build the annihilating polynomial.* Let $\nu_1,\dots,\nu_d$ be the distinct eigenvalues,
all positive since $A$ is positive definite, and set

$$
p(t) = \prod_{i=1}^{d}\left(1 - \frac{t}{\nu_i}\right) \in \mathcal{P}_d, \qquad p(0) = 1 .
$$

*Step 3 — it annihilates $A$.* $A$ is symmetric, hence orthogonally diagonalizable, and $p$
vanishes at every eigenvalue, so $p(A) = 0$.

*Step 4 — apply CG optimality.* Theorem 4.6 statement 4 gives

$$
\lVert e_d \rVert_A = \min_{q \in \mathcal{P}_d,\ q(0)=1} \lVert q(A)e_0 \rVert_A \le \lVert p(A)e_0 \rVert_A = 0 .
$$

*Step 5.* Therefore $e_d = 0$ with $d \le r+1$.

$$
\boxed{\text{CG terminates in at most } r+1 \text{ iterations}}
$$

**Key takeaway.** This is why preconditioners that capture a few dominant eigendirections work so
well: they need not shrink $\kappa$, only reduce the number of distinct eigenvalue clusters.

In [41]:
n_lr, r_lr = 50, 4
Wl = rng.standard_normal((n_lr, r_lr))
Wl, _ = np.linalg.qr(Wl)
K_lr = Wl @ np.diag(np.linspace(1.0, 5.0, r_lr)) @ Wl.T
A_lr = np.eye(n_lr) + K_lr
A_lr = (A_lr + A_lr.T) / 2
b_lr = rng.standard_normal(n_lr)
x_lr = np.linalg.solve(A_lr, b_lr)
lam_lr = np.linalg.eigvalsh(A_lr)
print("rank(K) =", np.linalg.matrix_rank(K_lr),
      "  distinct eigenvalues of A:", len(np.unique(np.round(lam_lr, 8))))
x = np.zeros(n_lr)
res = b_lr - A_lr @ x
p = res.copy()
rr = res @ res
for k in range(1, r_lr + 3):
    Ap = A_lr @ p
    al = rr / (p @ Ap)
    x = x + al * p
    res = res - al * Ap
    rrn = res @ res
    p = res + (rrn / rr) * p
    rr = rrn
    print(f"  k = {k}: relative error = {np.linalg.norm(x - x_lr)/np.linalg.norm(x_lr):.3e}")
    if k == r_lr + 1:
        assert np.linalg.norm(x - x_lr) / np.linalg.norm(x_lr) < 1e-12
print(f"kappa_2(A) = {np.linalg.cond(A_lr):.2f}, yet CG needed at most r + 1 = {r_lr+1} steps")

rank(K) = 4   distinct eigenvalues of A: 5
  k = 1: relative error = 2.064e-01
  k = 2: relative error = 3.457e-02
  k = 3: relative error = 1.721e-02
  k = 4: relative error = 7.526e-04
  k = 5: relative error = 5.482e-16
  k = 6: relative error = 6.044e-16
kappa_2(A) = 6.00, yet CG needed at most r + 1 = 5 steps


### Problem L3.6 — Young's relation and the optimal SOR spectrum

**Statement.** For a consistently ordered matrix with non-zero diagonal, derive Young's relation
$(\lambda + \omega - 1)^2 = \omega^2\mu^2\lambda$ between the SOR eigenvalues $\lambda$ and the
Jacobi eigenvalues $\mu$, and deduce
$\omega_{\mathrm{opt}} = 2/(1+\sqrt{1-\rho(G_{\mathrm{J}})^2})$ with
$\rho(G_{\omega_{\mathrm{opt}}}) = \omega_{\mathrm{opt}} - 1$.

**Intuition.** Consistent ordering lets the superdiagonal and subdiagonal parts be rescaled
against each other without changing a determinant, which links the two characteristic
polynomials.

**Solution.**

*Step 1 — the SOR characteristic equation.* $\lambda$ is an eigenvalue of $G_\omega$ exactly when

$$
\det\bigl(\lambda(D - \omega L) - (1-\omega)D - \omega U\bigr) = 0,
\quad \text{that is} \quad
\det\bigl((\lambda + \omega - 1)D - \omega(\lambda L + U)\bigr) = 0 .
$$

*Step 2 — the consistent-ordering lemma (cited).* For a consistently ordered $A$ and any
$s \neq 0$,

$$
\det\bigl( \alpha D - (sL + s^{-1}U) \bigr) = \det\bigl( \alpha D - (L+U) \bigr) .
$$

Source: Young, *Iterative Solution of Large Linear Systems*, Theorem 5.2.1; Varga,
*Matrix Iterative Analysis*, 2nd ed., Theorem 4.3.

*Step 3 — apply it.* For $\lambda \neq 0$ divide the bracket by $\omega\sqrt\lambda$ and take
$s = \sqrt\lambda$:

$$
\det\left( \frac{\lambda + \omega - 1}{\omega\sqrt{\lambda}} D - (L + U) \right) = 0 .
$$

*Step 4 — read off a Jacobi eigenvalue.* That is exactly the statement that
$\mu = \dfrac{\lambda + \omega - 1}{\omega\sqrt{\lambda}}$ is an eigenvalue of
$G_{\mathrm{J}} = D^{-1}(L+U)$. Squaring gives Young's relation.

*Step 5 — solve for $\lambda$.* Treat it as a quadratic in $\sqrt{\lambda}$:
$\lambda - (\omega\mu\sqrt{\lambda}) \cdot \sqrt{\lambda}\big/\sqrt{\lambda} \dots$ — concretely,
$\lambda^2 - (2(1-\omega) + \omega^2\mu^2)\lambda + (1-\omega)^2 = 0$, with discriminant

$$
\Delta = \omega^2\mu^2\bigl(\omega^2\mu^2 + 4(1-\omega)\bigr).
$$

*Step 6 — the optimum.* For $\Delta \ge 0$ the larger root exceeds $\lvert 1-\omega \rvert$ and
increases with $\mu$; for $\Delta \lt 0$ the two roots are complex conjugate with
$\lvert \lambda \rvert^2 = (1-\omega)^2$, so $\lvert \lambda \rvert = \omega - 1$ for
$\omega \gt 1$. The spectral radius is therefore minimized at the $\omega$ where the discriminant
first vanishes for $\mu = \rho(G_{\mathrm{J}})$:

$$
\omega^2\rho^2 + 4(1-\omega) = 0 \implies \omega_{\mathrm{opt}} = \frac{2}{1+\sqrt{1-\rho^2}} .
$$

*Step 7 — the optimal radius.* At that $\omega$ the double root is
$\lambda = \omega_{\mathrm{opt}} - 1$, so $\rho(G_{\omega_{\mathrm{opt}}}) = \omega_{\mathrm{opt}}-1$.

$$
\boxed{(\lambda + \omega - 1)^2 = \omega^2\mu^2\lambda, \qquad \omega_{\mathrm{opt}} = \frac{2}{1+\sqrt{1-\rho(G_{\mathrm{J}})^2}}, \qquad \rho(G_{\omega_{\mathrm{opt}}}) = \omega_{\mathrm{opt}} - 1}
$$

**Key takeaway.** Only the consistent-ordering lemma is quoted; everything else is algebra. On the
1-D Laplacian $\rho(G_{\mathrm{J}}) = \cos(\pi/(N+1))$, so $\rho(G_{\omega_{\mathrm{opt}}})$
behaves like $1 - 2\pi h$: SOR needs $O(N)$ iterations where Jacobi needs $O(N^2)$.

In [42]:
N_y = 16
A_y = (np.diag(2.0 * np.ones(N_y)) + np.diag(-np.ones(N_y - 1), 1)
       + np.diag(-np.ones(N_y - 1), -1))
D_y = np.diag(np.diag(A_y))
L_y = -np.tril(A_y, -1)
U_y = -np.triu(A_y, 1)
mu_y = max(abs(np.linalg.eigvals(np.linalg.solve(D_y, L_y + U_y))))
w_y = 2 / (1 + np.sqrt(1 - mu_y ** 2))
G_y = np.linalg.solve(D_y / w_y - L_y, (1 / w_y - 1) * D_y + U_y)
lam_y = np.linalg.eigvals(G_y)
print(f"rho(G_J) = {mu_y:.8f}   cos(pi/(N+1)) = {np.cos(np.pi/(N_y+1)):.8f}")
print(f"omega_opt = {w_y:.8f}   rho(G_omega) = {max(abs(lam_y)):.8f}   "
      f"omega_opt - 1 = {w_y-1:.8f}")
mus = np.linalg.eigvals(np.linalg.solve(D_y, L_y + U_y))
worst_match = max(min(abs((lam + w_y - 1) ** 2 - w_y ** 2 * mu ** 2 * lam) for mu in mus)
                  for lam in lam_y)
print("every SOR eigenvalue matches some Jacobi eigenvalue; worst residual:", worst_match)
assert worst_match < 1e-10
grid_w = np.linspace(1.0, 1.99, 3001)
rhos = [max(abs(np.linalg.eigvals(np.linalg.solve(D_y / w - L_y, (1 / w - 1) * D_y + U_y))))
        for w in grid_w]
print(f"grid minimizer omega = {grid_w[int(np.argmin(rhos))]:.6f}, rho = {min(rhos):.8f}")
assert abs(max(abs(lam_y)) - (w_y - 1)) < 1e-6
assert abs(grid_w[int(np.argmin(rhos))] - w_y) < 1e-2

rho(G_J) = 0.98297310   cos(pi/(N+1)) = 0.98297310
omega_opt = 1.68954662   rho(G_omega) = 0.68954662   omega_opt - 1 = 0.68954662
every SOR eigenvalue matches some Jacobi eigenvalue; worst residual: 2.0518486441330165e-15
grid minimizer omega = 1.689700, rho = 0.68970000


### Problem L3.7 — Randomized Kaczmarz converges at rate $1 - 1/\kappa_F^2$

**Statement.** Let $A \in \mathbb{R}^{m \times n}$ have full column rank and unit rows,
$\lVert a_i \rVert_2 = 1$. Randomized Kaczmarz picks a row $i$ with probability
$\lVert a_i \rVert_2^2 / \lVert A \rVert_F^2$ and sets $x_{k+1} = x_k + (b_i - a_i^{\top}x_k)a_i$.
Prove

$$
\mathbb{E}\bigl[\lVert x_{k+1} - x \rVert_2^2\bigr] \le \left(1 - \frac{1}{\kappa_F(A)^2}\right)\mathbb{E}\bigl[\lVert x_k - x \rVert_2^2\bigr],
\qquad \kappa_F(A) = \frac{\lVert A \rVert_F}{\sigma_{\min}(A)} .
$$

**Intuition.** Each step is an orthogonal projection onto one hyperplane, which removes exactly
the error component along that row; averaging over rows removes a fixed fraction of the whole
error.

**Solution.**

*Step 1 — the error recursion.* With $e_k = x_k - x$ and $b_i = a_i^{\top}x$,

$$
e_{k+1} = e_k - (a_i^{\top}e_k)a_i = (I - a_ia_i^{\top})e_k .
$$

*Step 2 — one step in norm.* Since $\lVert a_i \rVert_2 = 1$, $I - a_ia_i^{\top}$ is an
orthogonal projector, so

$$
\lVert e_{k+1}\rVert_2^2 = \lVert e_k \rVert_2^2 - (a_i^{\top}e_k)^{2} .
$$

*Step 3 — average over the row choice.* Unit rows give $\lVert A \rVert_F^2 = m$ and
$p_i = 1/m$, so

$$
\mathbb{E}\bigl[\lVert e_{k+1}\rVert_2^2 \mid e_k\bigr] = \lVert e_k \rVert_2^2 - \frac{1}{m}\sum_{i=1}^{m}(a_i^{\top}e_k)^2 = \lVert e_k \rVert_2^2 - \frac{\lVert Ae_k \rVert_2^2}{\lVert A \rVert_F^{2}} .
$$

*Step 4 — bound below by the smallest singular value.*
$\lVert Ae_k \rVert_2^2 \ge \sigma_{\min}(A)^2 \lVert e_k \rVert_2^2$, hence

$$
\mathbb{E}\bigl[\lVert e_{k+1}\rVert_2^2 \mid e_k\bigr] \le \left(1 - \frac{\sigma_{\min}(A)^2}{\lVert A \rVert_F^{2}}\right)\lVert e_k \rVert_2^2 .
$$

*Step 5 — take total expectations* and identify $\sigma_{\min}^2/\lVert A \rVert_F^2 = 1/\kappa_F^2$.

$$
\boxed{\mathbb{E}\bigl[\lVert e_{k+1}\rVert_2^2\bigr] \le \left(1 - \frac{1}{\kappa_F(A)^{2}}\right)\mathbb{E}\bigl[\lVert e_k \rVert_2^2\bigr]}
$$

**Key takeaway.** The rate depends on $\kappa_F$, not on $m$ or $n$ separately, so one pass over
a huge overdetermined system can be cheaper than forming $A^{\top}A$. This is the row-action
analogue of the stationary methods of Theorem 4.2.

In [43]:
m_kz, n_kz = 200, 20
A_kz = rng.standard_normal((m_kz, n_kz))
A_kz = A_kz / np.linalg.norm(A_kz, axis=1, keepdims=True)
x_kz = rng.standard_normal(n_kz)
b_kz = A_kz @ x_kz
sig_min = np.linalg.svd(A_kz, compute_uv=False)[-1]
kappa_F = np.linalg.norm(A_kz, "fro") / sig_min
rate = 1 - 1 / kappa_F ** 2
trials, steps = 40, 400
sq = np.zeros(steps + 1)
for _ in range(trials):
    x = np.zeros(n_kz)
    sq[0] += np.linalg.norm(x - x_kz) ** 2
    for k in range(steps):
        i = int(rng.integers(m_kz))
        x = x + (b_kz[i] - A_kz[i] @ x) * A_kz[i]
        sq[k + 1] += np.linalg.norm(x - x_kz) ** 2
sq /= trials
print(f"sigma_min = {sig_min:.6f},  ||A||_F = {np.linalg.norm(A_kz,'fro'):.6f},  "
      f"kappa_F = {kappa_F:.4f}")
print(f"predicted per-step factor  = {rate:.6f}")
print(f"observed per-step factor   = {(sq[steps]/sq[0])**(1/steps):.6f}")
print(f"E||e_400||^2 = {sq[-1]:.4e}   bound {sq[0]*rate**steps:.4e}")
assert (sq[steps] / sq[0]) ** (1 / steps) <= rate + 1e-6
assert sq[-1] <= sq[0] * rate ** steps

sigma_min = 2.238515,  ||A||_F = 14.142136,  kappa_F = 6.3176
predicted per-step factor  = 0.974945
observed per-step factor   = 0.955350
E||e_400||^2 = 3.2978e-07   bound 1.1098e-03


### Problem L3.8 — GMRES stagnation on a permutation matrix

**Statement.** Let $S$ be the $n \times n$ cyclic shift, $Se_j = e_{j+1}$ for $j \lt n$ and
$Se_n = e_1$, and let $b = e_1$, $x_0 = 0$. Prove that GMRES satisfies
$\lVert r_k \rVert_2 = 1$ for $k = 0, 1, \dots, n-1$ and $\lVert r_n \rVert_2 = 0$, and explain
which factor in Theorem 4.8 is responsible.

**Intuition.** The shift walks the residual around a cycle, so before the walk closes there is no
combination of the visited coordinates that cancels anything.

**Solution.**

*Step 1 — the Krylov space.* $S^{j}e_1 = e_{j+1}$ for $j \lt n$, so
$\mathcal{K}_k(S, e_1) = \operatorname{span}\{e_1,\dots,e_k\}$.

*Step 2 — the residual of a candidate.* Take $x = \sum_{j=1}^{k}c_je_j$ with $k \lt n$. Then
$Sx = \sum_{j=1}^{k}c_je_{j+1}$ and

$$
r = e_1 - Sx = e_1 - c_1e_2 - c_2e_3 - \dots - c_ke_{k+1}.
$$

*Step 3 — minimize.* The coordinates are distinct basis vectors, so
$\lVert r \rVert_2^2 = 1 + c_1^2 + \dots + c_k^2 \ge 1$, minimized at $c = 0$. Hence
$\lVert r_k \rVert_2 = 1$ for every $k \lt n$.

*Step 4 — step $n$.* $\mathcal{K}_n = \mathbb{R}^n$ and $S^{-1}e_1 = e_n$, so $x_n = e_n$ and
$r_n = 0$.

*Step 5 — which factor.* $S$ is a permutation matrix, so $S^{\top}S = SS^{\top} = I$: it is
orthogonal and therefore **normal**, and its eigenvector matrix (the Fourier matrix) is unitary,
so $\kappa_2(V) = 1$. The stagnation is entirely the spectral factor. Its eigenvalues are the
$n$-th roots of unity $\omega^{j}$, and for any $p \in \mathcal{P}_k^1$ with $k \lt n$,

$$
\frac{1}{n}\sum_{j=0}^{n-1} p(\omega^{j}) = p(0) \cdot 1 = 1
$$

because $\sum_j \omega^{jm} = 0$ for $1 \le m \le k \lt n$. A mean of modulus $1$ forces
$\max_j \lvert p(\omega^j) \rvert \ge 1$, so the minimax factor in Theorem 4.8 part 4 equals $1$.

$$
\boxed{\lVert r_k \rVert_2 = \lVert r_0 \rVert_2 \text{ for } k \lt n, \quad \lVert r_n \rVert_2 = 0; \text{ the cause is the spectrum, not non-normality}}
$$

**Key takeaway.** The common statement that GMRES stalls "because the matrix is non-normal" is
false here: $\kappa_2(V) = 1$ and the bound of Theorem 4.8 is tight. What matters is that the
eigenvalues surround the origin, so no low-degree polynomial normalized at $0$ can be small on
all of them. The genuinely non-normal failure mode is the defective matrix of Section 7.6.

In [44]:
for n_s2 in (5, 8):
    S2 = np.zeros((n_s2, n_s2))
    for i in range(n_s2):
        S2[(i + 1) % n_s2, i] = 1.0
    b_s2 = np.zeros(n_s2)
    b_s2[0] = 1.0
    res_s2 = gmres_res(S2, b_s2, n_s2)
    ev = np.linalg.eigvals(S2)
    print(f"n = {n_s2}: normal? ||S^T S - S S^T|| = {np.linalg.norm(S2.T@S2 - S2@S2.T):.1e}, "
          f"|eigs| = {np.round(np.abs(ev),10)[:3]} ...")
    print(f"        residuals {np.round(res_s2, 12)}")
    assert np.allclose(res_s2[:n_s2], 1.0) and res_s2[n_s2] < 1e-12
    # the mean-value argument, checked numerically
    for _ in range(50):
        k = int(rng.integers(1, n_s2))
        coef = rng.standard_normal(k)
        vals = np.array([1.0 + sum(coef[j] * z ** (j + 1) for j in range(k)) for z in ev])
        assert abs(vals.mean() - 1.0) < 1e-10
        assert np.abs(vals).max() >= 1.0 - 1e-10
print("mean-value identity and the resulting lower bound verified on random polynomials")

n = 5: normal? ||S^T S - S S^T|| = 0.0e+00, |eigs| = [1. 1. 1.] ...
        residuals [1. 1. 1. 1. 1. 0.]
n = 8: normal? ||S^T S - S S^T|| = 0.0e+00, |eigs| = [1. 1. 1.] ...
        residuals [1. 1. 1. 1. 1. 1. 1. 1. 0.]
mean-value identity and the resulting lower bound verified on random polynomials
